In [3]:
# from pathlib import Path
# import sys

# project_root = Path.cwd().parent  # notebooks -> project root
# sys.path.insert(0, str(project_root))

# from clinical_synopsis.embedder import Embedder
# print("embedder import OK")

# to avoid clinical_synopsis.embedder
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "clinical_synopsis"))

from embedder import Embedder
print("embedder import OK")

embedder import OK


# 0. Data in RETRIEVAL folder

Each file in `data/retrieval` supports one part of your RAG retrieval stack. 
- `minsearch_index.pkl` + `minsearch_documents.json` power **lexical search**.  
- `vector_index.npz` + `vector_index_metadata.json` power **semantic search**.  
- `metadata.db` is extra bookkeeping metadata about what you indexed.

Specifically:

- `minsearch_index.pkl` – the **lexical search index**  
  - A pickled Python object created by `minsearch.Index.fit(...)`.  
  - It contains the TF‑IDF / BM25–style structures that power your `search()` function.  
  - When you call `index = load_index()`, this is the file being loaded.

- `vector_index.npz` – the **embedding matrix and chunk IDs**  
  - A NumPy `.npz` archive with arrays like:
    - `embeddings`: a big 2D array, one embedding vector per chunk.
    - `chunk_ids`: the list of chunk IDs, aligned row‑by‑row with `embeddings`.  
  - When you do `vector_embeddings, vector_documents = load_vector_index()`, this file provides `vector_embeddings`.

- `vector_index_metadata.json` – the **metadata for each embedding row**  
  - JSON with a `documents` list. Each item is a dict for one chunk, with fields like:
    - `chunk_id`
    - `patient_id`
    - `doc_type`
    - `title`, `heading`
    - `is_oncology`
    - `chunk_text`, etc.  
  - `load_vector_index()` uses `chunk_ids` from `.npz` to reorder this list so `vector_documents[i]` matches `vector_embeddings[i]`.

- `minsearch_documents.json` – the **documents used to build the lexical index**  
  - JSON list of the chunk dictionaries that were fed into `minsearch.Index.fit(...)`.  
  - It’s basically the corpus for your lexical index, saved so you can inspect or rebuild it later.

- `metadata.db` – a **small database of higher-level document/patient metadata**  
  - Likely a SQLite DB (given the `.db` name) that stores additional information:
    - which files/patients were processed,
    - maybe source paths, dates, or other indexing metadata.  
  - It’s not used directly in `rag.py`, but is useful for bookkeeping and possibly other scripts in your project. (it is used by `build_minsearch_index.py` and `build_vector_index.py`)




In [4]:
!ls /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval

metadata.db		  minsearch_index.pkl  vector_index_metadata.json
minsearch_documents.json  vector_index.npz


## What to included on GitHub

What should go into `data/` on GitHub depends on whether you want the repo to be:

- **fully runnable immediately**, or
- **lighter-weight but rebuildable from code**.

For a RAG project, reproducibility usually means including either the derived retrieval artifacts or clear code and instructions to regenerate them. [github](https://github.com/gchure/reproducible_research)

## If you want it to run right away

Then yes, you should include the retrieval artifacts your `rag.py` directly depends on:

- `data/retrieval/minsearch_index.pkl`
- `data/retrieval/vector_index.npz`
- `data/retrieval/vector_index_metadata.json`

And likely also:

- `data/retrieval/minsearch_documents.json`
- `data/retrieval/metadata.db`

because they help inspection, rebuilding, or explainability.

If those files are present, someone cloning the repo can run `rag.py` without rebuilding the indexes first.

## Minimum needed by `rag.py`

Strictly speaking, your current `rag.py` only directly loads:

- `minsearch_index.pkl`
- `vector_index.npz`
- `vector_index_metadata.json`

So those three are the true runtime minimum for the current script.

## But should that be all in `data/`?

Probably not the entire `data/` folder. A cleaner GitHub setup is usually:

- keep only the **small, necessary, non-sensitive derived artifacts** that make the project runnable,
- exclude big temporary or rebuildable files,
- and explain in `README.md` how to regenerate indexes if needed. Good reproducible-research structure usually separates raw, derived, and scratch data and documents what each artifact is for. [audreyrpark.github](https://audreyrpark.github.io/RPl-Spielman-2020/template_readme.html)

## Practical recommendation for your project

For your course project, I would suggest:

### Include
- `data/retrieval/minsearch_index.pkl`
- `data/retrieval/vector_index.npz`
- `data/retrieval/vector_index_metadata.json`
- maybe `data/retrieval/minsearch_documents.json` if it is not too large

### Optional
- `metadata.db`, only if another script actually uses it or it helps explain the pipeline

### Also include
- the preprocessing/index-building scripts
- a short README note saying these are derived retrieval artifacts generated from your processed patient documents

## One thing to check first

Before pushing, check file sizes. GitHub is fine with normal files, but large binary artifacts can become awkward to version. If `vector_index.npz` is large, you may prefer to:
- leave it out,
- and provide a script to regenerate it instead.

So the short answer is:

- **No, not automatically “all that and only that.”**
- **Yes, those three retrieval files are the minimum needed for your current `rag.py` to run.**

A good next step is to list the sizes of those files and decide whether you want a “runs immediately” repo or a “rebuild first” repo.

# 1. Inspect retrieval of chunks with exact patient IDs

To make notebook testing easier, create a tiny helper cell to copy one exact patient ID from there each time to avoid patient id mismatch.
```
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
```
(**Where's vector_documents from?**)

You’re now at the point where it makes sense to compare:

- lexical,

- semantic,

- hybrid

for the same patient and same query. That is the right setup for the notebook experiment before you build the CSV evaluation runner.


`rag` is the module you imported with `import rag as rag`. It contains:
- vector_embeddings
- vector_documents
and the `rag()` function itself.

`rag.vector_documents` was created in the module by:
```python
vector_embeddings, vector_documents = load_vector_index()
```
`vector_documents` is a Python list of dictionaries, one per chunk, loaded from `vector_index_metadata.json` (in `data/retrieval/vector_index_metadata.json`). Each dict has metadata like:
- chunk_id
- patient_id
- doc_type
- title
- heading
- is_oncology
etc.

For testing
`rag.vector_documents[0]`
takes the first chunk in that list. That’s just an arbitrary but valid example chunk.

`rag.vector_documents[0]["patient_id"]`
reads the "patient_id" field from that first chunk’s metadata. This gives you a concrete patient ID string that you know exists in both:
- the lexical index (minsearch), and
- the vector index metadata.


So `vector_documents` is “the metadata that corresponds row‑by‑row to vector_embeddings, built using both the .npz file and its metadata JSON,” not as being loaded from the .npz alone.

In the folder `data/retrieval`:
- `vector_index.npz` holds the embedding matrix and the chunk ID list.
- `vector_index_metadata.json` holds the documents list with metadata, including patient_id.
Your `load_vector_index()` reads both and then wires them together: 
```python
def load_vector_index():
    data = np.load(VECTOR_INDEX_PATH, allow_pickle=True)
    with open(VECTOR_METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    embeddings = data["embeddings"]
    chunk_ids = data["chunk_ids"].tolist()
    documents = metadata["documents"]

    docs_by_chunk_id = {doc["chunk_id"]: doc for doc in documents}
    ordered_docs = [docs_by_chunk_id[chunk_id] for chunk_id in chunk_ids]

    return embeddings, ordered_docs
```
So:
- `vector_embeddings` comes from `vector_index.npz` (embeddings array).
- `vector_documents` comes from `vector_index_metadata.json` (`metadata["documents"]`), but reordered to match the embedding rows according to chunk_ids.
Conceptually, for your notebook:
- `vector_embeddings[i]` = embedding vector for chunk i, stored in vector_index.npz,
- `vector_documents[i]` = dict with patient_id, doc_type, etc. for the same chunk, coming from the JSON.

That's why pid is a real patient ID attached to the same chunk as `vector_embeddings[0]`.

In [5]:
import rag as rag

# Look at the first chunk we have in the vector index, take the patient_id attached to that chunk:
pid = rag.vector_documents[0]["patient_id"]

result = rag.rag(
    query="What oncology-related events are documented?",
    patient_id=pid,
    search_type="hybrid",
    num_results=5,
)

len(result["search_results"]), result["search_results"][:2]

(5,
 [{'id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'chunk_id': '6e3f2f8c188a50ec8d84a45b499a14a631fcc414',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': 'eacfd84f2024e811caae390e056a4e52fefc5249',
   'doc_type': 'oncology_timeline',
   'title': 'Oncology Timeline: Aurora248 Dooley940',
   'heading': 'Oncology Timeline: Aurora248 Dooley940',
   'chunk_text': '- Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18\n- Oncology-related dated events: 52',
   'chunk_index': 0,
   'is_oncology': '1',
   'date_start': '',
   'date_end': '',
   'rrf_score': 0.01639344262295082},
  {'chunk_id': 'f50d5c213d71cb54157be1dbe78d05ff2cb23489',
   'patient_id': '03b93198-d95e-c385-c3a7-80470f411d18',
   'document_id': '304ddea3daed73b3c0c0f47535fef1906ea22987',
   'doc_type': 'oncology_timeline_events',
   'title': 'oncology_timeline_events.csv',
   'heading': 'oncology_timeline_events',
   'chunk_text': "event_type: Observation; date: 2015-10-01T05:29:59-04:00; label:

In [6]:
print("Answer cost (USD):", result["answer_total_cost_usd"])
print("Eval cost (USD):  ", result["eval_total_cost_usd"])
print("Overall cost (USD):", result["overall_total_cost_usd"])

Answer cost (USD): 0.0028627500000000003
Eval cost (USD):   0.00220575
Overall cost (USD): 0.0050685


In [7]:
print("Answer tokens (in/out/total):",
      result["prompt_tokens"],
      result["completion_tokens"],
      result["total_tokens"])

print("Eval tokens (in/out/total):",
      result["eval_prompt_tokens"],
      result["eval_completion_tokens"],
      result["eval_total_tokens"])

Answer tokens (in/out/total): 1951 311 2262
Eval tokens (in/out/total): 2389 92 2481


In [8]:
for i, doc in enumerate(result["search_results"], start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Chunk ID:", doc.get("chunk_id"))
    print("Patient ID:", doc.get("patient_id"))
    print("Doc type:", doc.get("doc_type"))
    print("Title:", doc.get("title"))
    print("Heading:", doc.get("heading"))
    print("Text:", doc.get("chunk_text", "")[:500])

Rank: 1
Chunk ID: 6e3f2f8c188a50ec8d84a45b499a14a631fcc414
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Dooley940
Heading: Oncology Timeline: Aurora248 Dooley940
Text: - Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
- Oncology-related dated events: 52
Rank: 2
Chunk ID: f50d5c213d71cb54157be1dbe78d05ff2cb23489
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline_events
Title: oncology_timeline_events.csv
Heading: oncology_timeline_events
Text: event_type: Observation; date: 2015-10-01T05:29:59-04:00; label: Cancer Disease Progression; status: Patient's condition improved; resource_id: 3ee3c67c-a4a3-37e1-e49b-f20f32cc14db; source_file: data/prototype/sample50/Aurora248_Dooley940_03b93198-d95e-c385-c3a7-80470f411d18.json
Rank: 3
Chunk ID: 561c5809a7e9251705306dee52b08e9fd64721ce
Patient ID: 03b93198-d95e-c385-c3a7-80470f411d18
Doc type: oncology_timeline
Title: Oncology Timeline: Aurora248 Doole

In [9]:
available_patient_ids = sorted({doc["patient_id"] for doc in rag.vector_documents})
available_patient_ids[:10]

['03b93198-d95e-c385-c3a7-80470f411d18',
 '0c0f2095-e8ab-7ac4-6ef4-625748255480',
 '0f5704ee-b38b-5a68-449d-9c44806517d0',
 '188e1f01-15b7-d51b-c76d-bdd7772a10e9',
 '25197dc8-9425-1999-5914-f2171b0d4e32',
 '263375ec-5856-81b8-9e51-1cb8e8bcba30',
 '29f6beee-162f-0113-7884-72245814693f',
 '397b2de6-ccd8-858f-bf4a-b6fc379589bd',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 '3af995f1-02a5-07ee-5a7e-e2470a017f1e']

In [10]:
# # running the same query for just the first patient id in the list of available patient ids
# # which is what we did above with pid

# result = rag.rag(
#     query="What oncology-related events are documented?",
#     patient_id=available_patient_ids[0],
#     is_oncology=True,
#     search_type="hybrid",
#     num_results=5,
# )

# for i, doc in enumerate(result["search_results"], start=1):
#     print("=" * 80)
#     print("Rank:", i)
#     print("Doc type:", doc.get("doc_type"))
#     print("Title:", doc.get("title"))
#     print("Heading:", doc.get("heading"))
#     print("Chunk ID:", doc.get("chunk_id"))
#     print("Text:", doc.get("chunk_text", "")[:800])

Your test_cases now need to use real patient IDs from the index, not placeholder IDs. For example:

(Note for the below selection:
Rows for the second and third questions (is_oncology=None) can include any chunks, including ones where the is_oncology field is missing in your underlying data.)

In [11]:
# 3 test cases for the first patient id in the list of available patient ids
test_cases = [
    {
        "patient_id": available_patient_ids[0],
        "query": "What oncology-related events are documented?",
        "is_oncology": True,
    },
    {
        "patient_id": available_patient_ids[0],
        "query": "What recent conditions are documented?",
        "is_oncology": None, # so retrieval may pick chunks where is_oncology is 0, 1, or entirely absent.
    },
    {
        "patient_id": available_patient_ids[0],
        "query": "What medications are mentioned?",
        "is_oncology": None,
    },
]

A df or more debugging, so you can see:

- whether retrieval returned anything,

- what kind of document came first,

- and whether the answer quality changes across search types.

In [12]:
import pandas as pd

# this took 38s

rows = []

for case in test_cases:
    for search_type in ["lexical", "semantic", "hybrid"]:
        result = rag.rag(
            query=case["query"],
            patient_id=case["patient_id"],
            is_oncology=case["is_oncology"],
            search_type=search_type,
            num_results=5,
        )

        rows.append({
            "patient_id": case["patient_id"],
            "query": case["query"],
            "is_oncology": case["is_oncology"],
            "search_type": search_type,

            "answer": result["answer"],
            "relevance": result.get("relevance"),
            "groundedness": result.get("groundedness"),
            "response_time": result.get("response_time"),

            "n_search_results": len(result.get("search_results", [])),
            "top_doc_type": result["search_results"][0].get("doc_type")
                if result.get("search_results") else None,
            "top_title": result["search_results"][0].get("title")
                if result.get("search_results") else None,

            # Answer tokens and cost
            "answer_prompt_tokens": result.get("prompt_tokens"),
            "answer_completion_tokens": result.get("completion_tokens"),
            "answer_total_tokens": result.get("total_tokens"),
            "answer_input_cost_usd": result.get("answer_input_cost_usd"),
            "answer_output_cost_usd": result.get("answer_output_cost_usd"),
            "answer_total_cost_usd": result.get("answer_total_cost_usd"),

            # Eval tokens and cost
            "eval_prompt_tokens": result.get("eval_prompt_tokens"),
            "eval_completion_tokens": result.get("eval_completion_tokens"),
            "eval_total_tokens": result.get("eval_total_tokens"),
            "eval_input_cost_usd": result.get("eval_input_cost_usd"),
            "eval_output_cost_usd": result.get("eval_output_cost_usd"),
            "eval_total_cost_usd": result.get("eval_total_cost_usd"),

            # Combined cost
            "overall_total_cost_usd": result.get("overall_total_cost_usd"),
        })

df = pd.DataFrame(rows)
df

,patient_id,query,is_oncology,search_type,answer,relevance,groundedness,response_time,n_search_results,top_doc_type,...,answer_input_cost_usd,answer_output_cost_usd,answer_total_cost_usd,eval_prompt_tokens,eval_completion_tokens,eval_total_tokens,eval_input_cost_usd,eval_output_cost_usd,eval_total_cost_usd,overall_total_cost_usd
0,03b93198-d95e-c385-c3a7-80470f411d18,What oncology-related events are documented?,True,lexical,The oncology timeline documents these events:\...,RELEVANT,GROUNDED,4.729512,5,oncology_timeline,...,0.002148,0.001152,0.003300,3247,73,3320,0.002435,0.000329,0.002764,0.006064
1,03b93198-d95e-c385-c3a7-80470f411d18,What oncology-related events are documented?,True,semantic,The oncology-related events documented in the ...,RELEVANT,GROUNDED,3.455765,5,oncology_timeline_events,...,0.000842,0.000869,0.001710,1442,63,1505,0.001081,0.000284,0.001365,0.003075
2,03b93198-d95e-c385-c3a7-80470f411d18,What oncology-related events are documented?,True,hybrid,The oncology-related events documented in the ...,RELEVANT,GROUNDED,3.800643,5,oncology_timeline,...,0.001463,0.001409,0.002872,2391,68,2459,0.001793,0.000306,0.002099,0.004971
3,03b93198-d95e-c385-c3a7-80470f411d18,What recent conditions are documented?,None,lexical,The recent conditions documented in the **cond...,RELEVANT,PARTLY_GROUNDED,3.985629,5,conditions,...,0.001234,0.000770,0.002003,1943,132,2075,0.001457,0.000594,0.002051,0.004054
4,03b93198-d95e-c385-c3a7-80470f411d18,What recent conditions are documented?,None,semantic,The recent conditions documented in the **Pati...,RELEVANT,GROUNDED,3.599205,5,oncology_timeline,...,0.001064,0.001053,0.002117,1780,59,1839,0.001335,0.000266,0.001600,0.003718
5,03b93198-d95e-c385-c3a7-80470f411d18,What recent conditions are documented?,None,hybrid,Recent conditions documented in the **Patient ...,RELEVANT,GROUNDED,3.911117,5,conditions,...,0.001147,0.001206,0.002353,1924,75,1999,0.001443,0.000337,0.001780,0.004133
6,03b93198-d95e-c385-c3a7-80470f411d18,What medications are mentioned?,None,lexical,The medications mentioned in the **medications...,RELEVANT,GROUNDED,3.221606,5,medications,...,0.001274,0.000599,0.001873,1960,66,2026,0.001470,0.000297,0.001767,0.003640
7,03b93198-d95e-c385-c3a7-80470f411d18,What medications are mentioned?,None,semantic,The medications mentioned in the record includ...,RELEVANT,GROUNDED,3.934580,5,patient_overview,...,0.001345,0.000878,0.002222,2115,74,2189,0.001586,0.000333,0.001919,0.004142
8,03b93198-d95e-c385-c3a7-80470f411d18,What medications are mentioned?,None,hybrid,The medications mentioned in the record are:\n...,RELEVANT,GROUNDED,3.486814,5,medications,...,0.001371,0.000963,0.002334,2169,62,2231,0.001627,0.000279,0.001906,0.004240


In [13]:
df.groupby("search_type")["overall_total_cost_usd"].sum()


search_type
hybrid      0.013344
lexical     0.013758
semantic    0.010934
Name: overall_total_cost_usd, dtype: float64

In [14]:
df.groupby("search_type")[["answer_total_cost_usd", "eval_total_cost_usd"]].mean()

,answer_total_cost_usd,eval_total_cost_usd
search_type,,
hybrid,0.002520,0.001928
lexical,0.002392,0.002194
semantic,0.002017,0.001628


# 2. Evaluation

You can apply exactly the same pattern from the course: define a small ground-truth table for retrieval, then compute hit rate@K and MRR@K over that table for each search mode (lexical, semantic, hybrid). The only difference is what you use as the “relevant” item for each question in your EHR setting. Hit rate@K and MRR@K are standard retrieval metrics for RAG, so your approach generalizes directly.

1. Define retrieval ground truth
In the course you had something like:

- A table where each row has: query, relevant_doc_id.

For your project, a row can be:

- patient_id
- question
- one or more gold chunk IDs or gold doc identifiers (the chunks that definitely contain the answer)

Example schema (as a Pandas-like table):

- patient_id: "PATIENT_001"
- question: "What oncology-related events are documented?"
- gold_chunk_ids: ["chunk-123", "chunk-456"]

You can store this as JSON or CSV, e.g.:

```json
[
  {
    "patient_id": "PATIENT_001",
    "question": "What oncology-related events are documented?",
    "gold_chunk_ids": ["chunk-123", "chunk-456"]
  },
  {
    "patient_id": "PATIENT_001",
    "question": "What medications are mentioned?",
    "gold_chunk_ids": ["chunk-789"]
  }
]
```

2. Run retrieval only, per search mode
For each ground-truth row, you want:
- the list of retrieved chunk IDs for each search mode,
- in rank order.

Use your existing search functions directly (without calling rag()):
(This gives you a ranked list of chunk_ids for each (patient_id, question, search_type).)
```python
def get_ranked_chunk_ids(query, patient_id, search_type, k=10):
    if search_type == "lexical":
        results = rag.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = rag.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = rag.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be lexical, semantic, or hybrid")

    return [doc["chunk_id"] for doc in results]
```

3. Compute Hit rate@K and MRR@K
Recall the definitions:
- **Hit rate@K**: for each query, 1 if any relevant chunk is in the top K, else 0; average over queries.
- **MRR@K**: for each query, find the rank of the first relevant chunk in the top K, use 
1/rank; 0 if no relevant chunk in top K; average over queries.

In code (for your notebook):
```python
def eval_hit_mrr_for_mode(gt_rows, search_type, k=10):
    """gt_rows is a list of dicts with keys:
       - patient_id
       - question
       - gold_chunk_ids (list of strings)
    """
    hits = []
    reciprocal_ranks = []

    for row in gt_rows:
        patient_id = row["patient_id"]
        question = row["question"]
        gold_ids = set(row["gold_chunk_ids"])

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        # Hit@K
        hit = int(any(doc_id in gold_ids for doc_id in retrieved_ids))
        hits.append(hit)

        # MRR@K
        rr = 0.0
        for rank, doc_id in enumerate(retrieved_ids, start=1):
            if doc_id in gold_ids:
                rr = 1.0 / rank
                break
        reciprocal_ranks.append(rr)

    hit_rate = sum(hits) / len(hits) if hits else 0.0
    mrr = sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0.0
    return hit_rate, mrr
```

Then evaluate all three modes:
```python
search_types = ["lexical", "semantic", "hybrid"]

for st in search_types:
    hit_k, mrr_k = eval_hit_mrr_for_mode(gt_rows, st, k=5)
    print(f"{st}: Hit@5={hit_k:.3f}, MRR@5={mrr_k:.3f}")
```

This replicates the course pattern: for each “ground-truth record” (your patient–question–gold evidence row), you get a retrieval relevance score per search mode.

4. Use the metrics for tuning
Once you have Hit@K and MRR@K per search mode, you can use them exactly as in the course to tune search parameters, for example:
- in `search()`: adjust `boost_dict` weights (`title`, `heading`, `chunk_text`),
- in `semantic_search()`: adjust normalization or top‑N cutoff,
- in `hybrid_search()`: adjust `rrf_k`, adjust the number of lexical vs semantic candidates you pass into RRF.

The evaluation workflow is:

1) Define or update the ground-truth rows.

2) Run lexical / semantic / hybrid retrieval on those rows.

3) Compute Hit@K and MRR@K.

4) Tweak parameters and re-measure.

5) Pick the setting that gives the best metrics for your chosen K (e.g. K=5).


You’re right that in the course the “ground truth” basically covers the whole tiny toy dataset (72 docs × 5 questions = 360 query–label pairs). That’s just a convenient teaching setup, not a rule you must follow. In your project you usually only label a small, representative subset of your full EHR dataset.

What “ground truth” is for here
For your RAG/search evaluation, the ground truth is just:
- a set of queries (your questions), and
- for each query, the documents/chunks you mark as relevant.
Its job is to estimate how good your retrieval is, not to cover every single record the app will ever see.

**How big should your ground truth be?**
There is no fixed number. Typical guidance for ranking/RAG evaluation is:
- Start with something like 50–100 queries that:
    - cover the key use cases,
    - vary in difficulty and phrasing,
    - touch different types of documents/sections.
- If metrics are very noisy (change a lot when you tweak parameters), expand to more queries until performance curves stabilize.

For your project, a very sensible starting point is something like:
- pick, say, 5–10 patients you know well,
- for each, define 3–5 questions that reflect your app’s goals (e.g., “oncology history”, “medications”, “smoking status”),
- for each question, label one or a small set of gold chunks.
That already gives you 15–50 query–label pairs, enough to compare lexical vs semantic vs hybrid and do parameter tuning.

You already have a defined “population” of 50 patients; now you just need to choose a small subset of them (5–10) to label and evaluate on. That subset should be either random or deliberately representative.

1. What you’re sampling from
data/processed/mcode_breast_sample_50_manifest.csv is your list of 50 patients and their files. Conceptually:
- each row ≈ one patient,
- with columns like patient_id, file paths, etc.
Your ground-truth patients will be a subset of these rows; the full retrieval index can still include all 50.

2. Option A: simple random sample of 5–10 patients
If you don’t need special coverage (e.g., early vs late stage, different note types), the simplest is:
- Load the manifest CSV (e.g., in a notebook).
- Randomly sample, say, 8 patients.
- Use those patient IDs when you build your question–gold-chunk ground truth.
This is statistically clean and easy to explain in your report:
“From the 50 patients, we randomly selected 8 for manual ground-truth labeling and retrieval evaluation.”
```python
import pandas as pd

manifest_path = "data/processed/mcode_breast_sample_50_manifest.csv"
df = pd.read_csv(manifest_path)

# Sample 8 patients without replacement
sampled = df.sample(n=8, random_state=42)

sampled_patient_ids = sampled["patient_id"].tolist()
print(sampled_patient_ids)
```

3. Option B: deliberately pick “interesting” patients
Because your dataset is small, you might prefer to choose patients that:
- have oncology-related notes vs non-oncology notes,
- vary in number of documents or events,
- contain the kinds of information you care about (e.g., medications, treatments, staging).

A pragmatic recipe:
    1. Load the manifest and maybe join it with a summary of per-patient stats (e.g., count of chunks, presence of oncology flag).
    2. Manually inspect a few rows (or your CSV summaries) and pick:
        - 2–3 patients with many notes,
        - 2–3 with few notes,
        - 1–2 edge cases (e.g., unusual treatments).

You can document this as:
“We selected 7 patients to cover a range of note volumes and oncology characteristics.”

The trade-off:
- random sampling is cleaner for “unbiased” evaluation,
- hand-picking can ensure you actually exercise the key behaviors your app should support.

4. How this connects to ground truth size
Once you have your 5–10 patient IDs:
- For each selected patient, create 3–5 realistic questions.
- For each question, label the gold chunk_id(s).
That will give you somewhere around 15–50 query–label pairs.

You still index all 50 patients for retrieval, but you only evaluate metrics on that labeled subset. That’s exactly the intended pattern: a big searchable corpus, small labeled ground truth.

For option B, the best first step is a notebook cell that helps you inspect the manifest, summarize it per patient, and surface candidate patients to choose from. Since your goal is to pick “interesting” patients rather than sample randomly, you want code that shows things like number of files/rows per patient, note types, and any obviously useful columns.

In [15]:
# #we see that for every patient there is just 1 row

# import pandas as pd
# from IPython.display import display

# manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# # Load manifest
# df = pd.read_csv(manifest_path)

# print("Shape:", df.shape)
# print("\nColumns:")
# print(df.columns.tolist())

# print("\nFirst 5 rows:")
# display(df.head())

# print("\nMissing values per column:")
# display(df.isna().sum().sort_values(ascending=False))

# # Try to infer the patient column
# candidate_patient_cols = [c for c in df.columns if "patient" in c.lower() or c.lower().endswith("_id")]
# print("\nPossible patient ID columns:", candidate_patient_cols)

# # Pick the first likely patient ID column; change manually if needed
# if not candidate_patient_cols:
#     raise ValueError("Couldn't infer a patient ID column. Please inspect df.columns and set patient_col manually.")

# patient_col = candidate_patient_cols[0]
# print("\nUsing patient column:", patient_col)

# # Basic per-patient summary
# patient_summary = (
#     df.groupby(patient_col)
#       .agg(
#           n_rows=(patient_col, "size"),
#           n_unique_values=("source_file", "nunique") if "source_file" in df.columns else (patient_col, "size")
#       )
#       .sort_values("n_rows", ascending=False)
#       .reset_index()
# )

# # Add quick summaries for potentially useful columns
# useful_cols = [
#     c for c in df.columns
#     if c.lower() in {"doc_type", "document_type", "category", "section", "title", "source_file", "file_path"}
# ]

# for col in useful_cols:
#     if col != patient_col:
#         top_vals = (
#             df.groupby(patient_col)[col]
#               .apply(lambda s: ", ".join(map(str, s.dropna().astype(str).value_counts().head(3).index.tolist())))
#               .rename(f"top_{col}")
#         )
#         patient_summary = patient_summary.merge(top_vals, on=patient_col, how="left")

# print("\nPatients with the most rows/files:")
# display(patient_summary.head(15))

# print("\nPatients with the fewest rows/files:")
# display(patient_summary.tail(15))

# # Optional: inspect one patient in detail
# example_patient = patient_summary.iloc[0][patient_col]
# print(f"\nExample patient for closer inspection: {example_patient}")

# patient_view = df[df[patient_col] == example_patient].copy()
# display(patient_view.head(20))

# # Optional: if you want a mixed selection strategy, take:
# # - top 3 most complex patients
# # - bottom 2 simplest patients
# # - 3 from the middle
# n = len(patient_summary)
# mixed_candidates = pd.concat([
#     patient_summary.head(3),
#     patient_summary.iloc[max(n//2 - 1, 0): min(n//2 + 2, n)],
#     patient_summary.tail(2)
# ]).drop_duplicates(subset=[patient_col])

# print("\nSuggested mixed candidate set:")
# display(mixed_candidates)

# 9 patients for ground truth

We pick 9 patients with 3 from each complexity bucket (low, medium, high) for the ground-truth set, in order to cover simple, medium, and complex EHRs.

`n_resources` is the total number of FHIR resources in that patient’s bundle — i.e., how many individual clinical records (Patient, Encounter, Observation, Condition, Procedure, MedicationRequest, DiagnosticReport, etc.) are contained in the JSON file for that patient.

So we see that the higher `n_resources`, the higher `complexity_score`

Remember:
### Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.



In [58]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [17]:
df

,filename,patient_id,patient_name,n_resources,n_encounters,n_observations,n_conditions,n_procedures,n_medication_requests,n_medication_administrations,n_diagnostic_reports,first_date,last_date,followup_days,complexity_score,complexity_bucket,sample_seed
0,data/raw/longitudinalMCODEBreast/Corrie32_Boyl...,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,230,27,80,2,27,20,0,28,2020-12-18T03:22:18-05:00,2022-05-20T18:36:55-04:00,518,317,low,42
1,data/raw/longitudinalMCODEBreast/Florine959_St...,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,261,25,119,5,23,11,0,26,2019-06-11T21:25:37-04:00,2022-06-21T05:03:45-04:00,1105,330,low,42
2,data/raw/longitudinalMCODEBreast/Deana43_Baumb...,3f130449-d7db-f118-5bd0-cce81084e911,Deana43 Baumbach677,431,42,206,10,23,18,0,44,2012-12-18T03:09:54-05:00,2022-04-12T04:24:54-04:00,3402,540,low,42
3,data/raw/longitudinalMCODEBreast/Joni720_Stied...,43c173b0-172c-f414-5c62-1bdf4bb33954,Joni720 Stiedemann542,459,47,218,17,29,5,0,51,2014-04-22T07:35:47-04:00,2022-05-17T21:08:31-04:00,2947,579,low,42
4,data/raw/longitudinalMCODEBreast/Santos184_Jas...,64ae3769-65e4-222e-6793-1a3bc14ec682,Santos184 Jaskolski867,468,45,241,4,31,11,0,48,2010-08-02T09:36:48-04:00,2022-05-17T22:36:39-04:00,4306,584,low,42
5,data/raw/longitudinalMCODEBreast/Mónica985_Se...,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,436,61,151,6,54,8,0,62,2018-01-07T15:16:05-05:00,2022-04-21T02:52:37-04:00,1564,602,low,42
6,data/raw/longitudinalMCODEBreast/Maryellen651_...,af3bd539-de27-28d9-9016-f1643d4615c0,Maryellen651 Zboncak558,514,54,238,11,35,18,0,56,2011-08-29T07:05:19-04:00,2022-04-27T18:02:46-04:00,3894,660,low,42
7,data/raw/longitudinalMCODEBreast/Darcie474_Fra...,568ec0af-94fa-521b-012e-88f61f78028f,Darcie474 Frami345,496,69,180,2,59,11,0,71,2015-10-27T05:01:54-04:00,2022-04-25T17:02:50-04:00,2372,686,low,42
8,data/raw/longitudinalMCODEBreast/Jani266_Thiel...,3e693a9a-de55-de2e-aab9-500036bcf04b,Jani266 Thiel172,629,76,258,8,67,14,0,80,2011-05-02T16:40:40-04:00,2022-06-03T06:55:38-04:00,4049,844,low,42
9,data/raw/longitudinalMCODEBreast/Dolores502_Ca...,6fb374e8-33aa-a5ea-f050-b61394dfcb99,Dolores502 Caldera106,872,101,305,13,165,18,1,111,2006-05-18T17:21:05-04:00,2022-06-30T17:36:05-04:00,5887,1244,low,42


In [59]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


So we have a subset of 9 patients with a range of difficulty, which is exactly what you want when tuning retrieval parameters using Hit@K and MRR rather than evaluating only easy or only dense cases.



Pick a K that matches your prompt budget (e.g., if you typically feed 5 chunks into the LLM, use K=5)

In [20]:
# import math

# # Example: ground truth rows — fill these manually once you know chunk_ids
# gt_rows = [
#     # Replace with your real patient/question/gold_chunk_ids
#     {
#         "patient_id": "PATIENT_001",
#         "question": "Summarize the patient's oncology history.",
#         "gold_chunk_ids": ["chunk-id-1", "chunk-id-2"],
#     },
#     {
#         "patient_id": "PATIENT_002",
#         "question": "What medications is the patient taking?",
#         "gold_chunk_ids": ["chunk-id-10"],
#     },
#     # ...
# ]

# def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
#     """Run the chosen search and return top-k chunk_ids in rank order."""
#     if search_type == "lexical":
#         results = search(
#             query=query,
#             patient_id=patient_id,
#             num_results=k,
#         )
#     elif search_type == "semantic":
#         results = semantic_search(
#             query=query,
#             patient_id=patient_id,
#             num_results=k,
#         )
#     elif search_type == "hybrid":
#         results = hybrid_search(
#             query=query,
#             patient_id=patient_id,
#             num_results=k,
#         )
#     else:
#         raise ValueError("search_type must be one of: lexical, semantic, hybrid")

#     return [doc["chunk_id"] for doc in results]

# def hit_rate_at_k(retrieved_ids, relevant_ids, k):
#     """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
#     relevant = set(relevant_ids)
#     top_k = retrieved_ids[:k]
#     return int(any(doc_id in relevant for doc_id in top_k))

# def mrr_at_k(retrieved_ids, relevant_ids, k):
#     """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
#     relevant = set(relevant_ids)
#     for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
#         if doc_id in relevant:
#             return 1.0 / rank
#     return 0.0

# def eval_search_type(gt_rows, search_type, k=5):
#     hits = []
#     mrrs = []

#     for row in gt_rows:
#         patient_id = row["patient_id"]
#         question = row["question"]
#         gold_ids = row["gold_chunk_ids"]

#         retrieved_ids = get_ranked_chunk_ids(
#             query=question,
#             patient_id=patient_id,
#             search_type=search_type,
#             k=k,
#         )

#         hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
#         mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

#     hit_rate = sum(hits) / len(hits) if hits else math.nan
#     mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

#     return hit_rate, mrr

# # Evaluate lexical, semantic, and hybrid for K=5
# for st in ["lexical", "semantic", "hybrid"]:
#     hr, mrr = eval_search_type(gt_rows, st, k=5)
#     print(f"{st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

# Finding gold chunks

You choose `gold_chunk_ids` by **looking at your chunks for each patient and question and explicitly marking the ones that contain the evidence you want the retriever to find**, and you test enough questions to cover your main use cases without making labeling intractable — typically on the order of a few dozen, not hundreds. [linkedin](https://www.linkedin.com/posts/dan-bucureanu_here-are-the-easy-steps-to-perform-your-rag-activity-7341344334752985089-537X)

Here’s a practical way to do it for your 9 patients.

***

## 1. How to decide on gold_chunk_ids

For each selected patient and question:
See below for [Selecting questions](#selecting_questions)


1. **Pick a realistic clinical question**  
   Examples:
   - “Summarize the patient’s oncology history.”
   - “What treatments has the patient received?”
   - “What medications is the patient currently taking?”
   - “Does the patient have documented smoking status?”

2. **Use your retrieval or direct DB/CSV inspection to find candidate chunks**  
   - Run `search()` or `semantic_search()` for that patient and question and inspect the top K chunks.  
   - Or query `metadata.db` / look at the derived `.md`/`.csv` files directly.

3. **Mark the chunks that truly contain needed answer information**  
   For example:
   - For “oncology history,” the chunk(s) that list cancer diagnoses, staging, key events.
   - For “medications,” the chunk(s) that list drugs, doses, start dates.

4. **Add all such chunks to `gold_chunk_ids`**  
   - If one chunk has almost everything, you might have a single ID.
   - If the answer is spread across multiple sections (e.g., one chunk for diagnosis, one for treatment), include both. [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

Guidelines:

- Think of them as **“gold nugget” chunks**: if the retriever gets these, the LLM has enough factual material to answer correctly. [linkedin](https://www.linkedin.com/posts/dan-bucureanu_here-are-the-easy-steps-to-perform-your-rag-activity-7341344334752985089-537X)
- Don’t list *every* chunk that mentions the topic; focus on those that are clearly useful and needed.
- It’s fine if some questions have 1 gold chunk and others have 2–3; Hit@K and MRR work with sets of relevant items. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

***

## 2. How many questions to test

There’s no hard rule, but common practice for a small project like yours:

- **Per patient**: 3–5 questions that reflect your app’s intended use:
  - 1–2 on oncology history/events.
  - 1 on treatments/procedures.
  - 1 on medications.
  - 1 on another key aspect you care about (e.g., encounters timeline, labs). [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

- For **9 patients**:
  - If you do 3 questions each → ~27 questions.
  - If you do 4 each → ~36 questions.

That’s usually enough to:

- get reasonably stable Hit@K and MRR estimates,
- compare lexical vs semantic vs hybrid search,
- and tune a few parameters (boosts, `rrf_k`, K) without your labeling workload exploding. [dataaihub](https://www.dataaihub.co/learn/retrieval-evaluation)

Try to ensure your question set:

- covers different document types (`patient_overview`, `oncology_timeline`, `conditions`, `medications`, etc.),
- includes both easy and harder questions (some with answers in one chunk, some needing multiple chunks),
- includes both oncology and non-oncology questions so the `is_oncology` metadata actually matters. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

***

So a good target for your project is:

- 9 patients × 3–4 questions each → about 30 labeled question–gold-chunk sets,  
- with gold chunks chosen as the specific pieces of evidence you’d want the retriever to find for that question.

If you tell me one patient ID and one concrete question you care about, I can walk you through how to find and label gold_chunk_ids for that specific case.



# Let's try 1 patient
'41681ed6-efc5-94c0-1bc0-f60b34dbd31b':

Look at `data/interim/sample50/41681ed6-efc5-94c0-1bc0-f60b34dbd31b` - NO! Not interim, but DERIVED documents are the mock EHRS!

Just looking at the documents I don't see any oncology related events, which turns out to be wrong because she had at least two cancers, leukemia in 1982 and breast cancer in 2021 (doc_type: oncology_timeline - this is not a file in the interim folder, because it gets genrated for the derived/mock EHRs, where oncology_timeline is conditionally generated if there are 3 oncology related events.)

The folder contains `conditions.csv` where the columns `display` and `text` contain
- "Acute myeloid leukemia, disease (disorder)","Acute myeloid leukemia, disease (disorder)"
- "Malignant neoplasm of breast (disorder),Malignant neoplasm of breast (disorder)"
among many other disorders (there are also findings).

In [40]:
!ls /Users/barbarato/llm-zoomcamp-capstone-project/data/interim/sample50/41681ed6-efc5-94c0-1bc0-f60b34dbd31b

bundle_metadata.json	encounters.csv			observations.csv
conditions.csv		medication_administrations.csv	patient.csv
diagnostic_reports.csv	medication_requests.csv		procedures.csv


In [42]:
!ls /Users/barbarato/llm-zoomcamp-capstone-project/data/derived/sample50/41681ed6-efc5-94c0-1bc0-f60b34dbd31b

conditions.csv		medications.csv		      patient_overview.md
diagnostic_reports.csv	observations.csv	      procedures.csv
encounters.csv		oncology_timeline.md
manifest.json		oncology_timeline_events.csv



If there were NO CANCERS for this patient:
For this patient, “Summarize the patient’s oncology history” is actually a useful **negative** test case: if there truly is no cancer-related information, your gold set for that question should be **empty**, and a good retrieval + RAG pipeline should retrieve nothing clearly oncology-related and answer “I don’t know” or “No oncology history documented.” [pmc.ncbi.nlm.nih](https://pmc.ncbi.nlm.nih.gov/articles/PMC4457181/)

Here’s how to handle it.

***

1. Confirm there is no oncology content

Since your ingestion and chunking pipeline already tags oncology text (using `ONCOLOGY_TERMS` and `is_oncology` flags), you can double-check from `metadata.db` that this patient really has no oncology chunks:

In a notebook: ...

If `len(onc_chunks)` is 0 and your visual inspection of `data/interim/sample50/...` matches that, you can treat this patient as “no oncology history documented” for the purpose of this test. 

***

2. Define ground truth for this negative question

For the question:

> “Summarize the patient’s oncology history.”

your ground-truth row would look like:

```python
{
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": [],  # empty list on purpose
}
```

Interpretation:

- There are **no chunks** that contain oncology history for this patient.
- That’s valid: some questions have no relevant evidence in the corpus. [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

When you compute Hit@K and MRR:

- Hit@K for this question should be 0 if retrieval correctly avoids oncology-marked chunks.
- MRR@K will be 0 because there is no relevant chunk to find.

This kind of negative case is useful to:

- check that your `is_oncology` filters and lexical/semantic search don’t invent cancer history where none exists;
- evaluate whether the LLM respects the “If the answer is not found in the context, respond with 'I don't know.'” instruction. [medium](https://medium.com/@akankshasinha247/rag-series-part-5-evaluating-rag-pipelines-for-groundedness-relevance-and-factuality-d67031b18d0b)

***

3. How many questions like this?

For your overall ground truth:

- include **a few negative questions** like this (no gold chunks), for patients without oncology or without a particular attribute;
- most questions should still have 1–3 gold chunks (positive cases), so your Hit@K and MRR measure “did we find the right evidence when it exists?” [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

## A good mix of positive and negative questions for your 9 patients might be:

- 2–3 oncology questions per patient who actually has cancer-related data (positive questions),
- 1–2 “no oncology history” or “no X documented” questions for patients where that’s true (negative questions).

***

So for this specific patient and question:

- it’s a **negative ground-truth row** with `gold_chunk_ids` = `[]`,
- and it will help you evaluate whether retrieval and generation correctly say “there is no oncology history documented” rather than hallucinate one.

## Find oncology chunks

For patient '41681ed6-efc5-94c0-1bc0-f60b34dbd31b':

Number of oncology chunks: 116

In [21]:
# double-check from metadata.db that this patient really has no oncology chunks:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_id = "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

conn = sqlite3.connect(db_path)

chunks_df = pd.read_sql_query(
    """
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id = ?
    """,
    conn,
    params=[patient_id],
)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,973694e5061b9103d1b1da41a6fb44459e220b91,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,55b7724042878660090ad81894f826b359e296fe,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,258c58b10c5ee39c96b9b4f2d09735a50312bd14,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
3,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,8054ebc26affb0a53d95d7436d795a985dc4ef9b,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,b60783b603a902e86bac07eaef38c50509a72cc0,0,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...


Number of rows: 1641


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
6,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,d0c6336772cc8ee0cc79cc2628bdba3218017dc8,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
52,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,conditions.csv,conditions,d1ce3fd72756c7deeb385af432c9d7f00ae6bfef,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1084,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,observations,observations.csv,observations,cea4ce538b38f22f5530ac5dee22b00ba4cec3a3,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1085,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,observations,observations.csv,observations,00ec62530719197e41f04fb4c867bc31ca526aed,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1086,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,observations,observations.csv,observations,0093b36e5a57e2ba51da175385855bb663e02a7e,1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...


Number of oncology chunks: 116


In [43]:
# For this patient
onc_chunks_patient = onc_chunks  # already filtered by patient_id

# Look at a handful first to see the pattern
display(onc_chunks_patient[["doc_type", "title", "heading", "chunk_id", "chunk_text"]].head(20))

,doc_type,title,heading,chunk_id,chunk_text
6,conditions,conditions.csv,conditions,d0c6336772cc8ee0cc79cc2628bdba3218017dc8,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
52,conditions,conditions.csv,conditions,d1ce3fd72756c7deeb385af432c9d7f00ae6bfef,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1084,observations,observations.csv,observations,cea4ce538b38f22f5530ac5dee22b00ba4cec3a3,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1085,observations,observations.csv,observations,00ec62530719197e41f04fb4c867bc31ca526aed,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1086,observations,observations.csv,observations,0093b36e5a57e2ba51da175385855bb663e02a7e,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1087,observations,observations.csv,observations,3c9737d32d2ee8aa6c6b829261d331c1f1af03e1,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1088,observations,observations.csv,observations,d54578d00aadb8922d938b2f9f54f8abc5d4a6be,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1090,observations,observations.csv,observations,9739bb9966595b8eaf459acd31f4d82fb2f13abc,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1094,observations,observations.csv,observations,da690e20c4e69d15ff15f5b46eb33a9f6cb33e1d,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...
1095,observations,observations.csv,observations,9161513ac46e5ed5a974e84cc79620f30cfffb40,patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd3...


In [ ]:
onc_chunks_patient

In [ ]:
# there are 9 chunks for oncology_timeline and 2 for conditions
onc_chunks_patient["doc_type"].value_counts()

doc_type
oncology_timeline_events    49
procedures                  36
observations                11
oncology_timeline            9
patient_overview             9
conditions                   2
Name: count, dtype: int64

That’s actually great: it means this patient is a **positive oncology case**, so “Summarize the patient’s oncology history” is a very suitable evaluation question, and you’ll be able to define meaningful `gold_chunk_ids` for it.

Here’s how to proceed.

***

## 1. Use those 116 oncology chunks to pick gold ones

Right now, `onc_chunks` holds all chunks for this patient with `is_oncology == 1`. Among those 116, you want to identify the **core evidence** for an oncology history summary:

- chunks that describe:
  - cancer diagnoses (types, sites, staging),
  - key events (diagnosis date, recurrences, metastases),
  - treatments (surgery, chemo, radiation, targeted therapy),
  - perhaps important diagnostic reports and timelines. [pmc.ncbi.nlm.nih](https://pmc.ncbi.nlm.nih.gov/articles/PMC4457181/)

In your notebook, you can inspect them like:

```python
# For this patient
onc_chunks_patient = onc_chunks  # already filtered by patient_id

# Look at a handful first to see the pattern
display(onc_chunks_patient[["doc_type", "title", "heading", "chunk_id", "chunk_text"]].head(20))
```

Then:

- Scan for chunks in `doc_type` like `oncology_timeline`, `oncology_timeline_events`, `conditions`, `diagnostic_reports`, `procedures` — those are likely to contain the most useful oncology history.  
- Within those, read `chunk_text` to find chunks that:
  - talk about diagnosis and staging,
  - list major treatments and dates,
  - summarize progression.

Pick **a small set** (say 2–5) that together give you enough information to answer:

> “Summarize the patient’s oncology history.”

Those chunk IDs become your `gold_chunk_ids`.

***

## 2. Add these multiple chunks to your ground truth

For example, suppose you identify:

```python
gold_ids_oncology_history = [
    "chunk-id-diagnosis",
    "chunk-id-timeline",
    "chunk-id-treatment",
]
```

Then your row would be:

```python
{
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": gold_ids_oncology_history,
}
```

Now this question:

- is a **positive** case with multiple relevant chunks,
- and your Hit@K / MRR will measure whether retrieval brings **at least one of these chunks** near the top. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

--> I first chose just 2 chunks and only 1 made it to rank 13 in lexical search

***

## 3. Why this is useful for tuning

Questions like this are particularly valuable because:

- they exercise both **lexical** (exact term matches like “cancer”, “stage II”) and **semantic** retrieval (synonyms, varied phrasing), [arxiv](https://arxiv.org/html/2603.03541v1)
- they rely heavily on `doc_type`, `is_oncology`, and date metadata — exactly the metadata you worked to preserve in your ingestion pipeline. [codersarts](https://www.codersarts.com/post/building-a-metadata-aware-ingestion-retrieval-pipeline)
- they show you how well hybrid search and filters work across complex, multi-chunk answers.

If you’d like, next step could be to pick one specific chunk from `oncology_timeline` and one from `conditions` for this patient and I can help you turn them into concrete `gold_chunk_ids` for your ground truth.

Perfect; those two chunk IDs are exactly the kind of “gold nuggets” you want for your oncology-history question.

For this patient:

- `4412a94aa2cefe467b43d2f7588722ab03167c6e` – a chunk from `oncology_timeline` that describes her cancer.  
- `d0c6336772cc8ee0cc79cc2628bdba3218017dc8` – a chunk from `conditions` describing the cancer diagnosis.  

Together, they give enough evidence to answer:

> “Summarize the patient’s oncology history.”

So you can define your ground-truth row like:

```python
gt_row_oncology_history = {
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": [
        "4412a94aa2cefe467b43d2f7588722ab03167c6e",
        "d0c6336772cc8ee0cc79cc2628bdba3218017dc8",
    ],
}
```

Then:

- Hit@K for this question is 1 if **either** of these chunk IDs appears in the top K retrieved results;  
- MRR@K uses the **rank of the first of these two** that appears in the top K. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

You can add `gt_row_oncology_history` to your `gt_rows` list and start using it to compare lexical vs semantic vs hybrid retrieval for this patient.

# Test RAG with 2 chunks for this 1 patient, 1 question

## 2 chunks are not enough, as it will turn out below

In [22]:
gt_row_oncology_history = {
    "patient_id": "41681ed6-efc5-94c0-1bc0-f60b34dbd31b",
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": [
        "4412a94aa2cefe467b43d2f7588722ab03167c6e",
        "d0c6336772cc8ee0cc79cc2628bdba3218017dc8",
    ],
}

In [23]:
import math
import rag as rag

def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
    """Run the chosen search and return top-k chunk_ids in rank order."""
    if search_type == "lexical":
        results = rag.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = rag.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = rag.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    return [doc["chunk_id"] for doc in results]

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
    relevant = set(relevant_ids)
    top_k = retrieved_ids[:k]
    return int(any(doc_id in relevant for doc_id in top_k))

def mrr_at_k(retrieved_ids, relevant_ids, k):
    """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def eval_search_type(gt_row_oncology_history, search_type, k=5):
    hits = []
    mrrs = []

    for row in gt_row_oncology_history:
        patient_id = row["patient_id"]
        question = row["question"]
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

    return hit_rate, mrr

# Evaluate lexical, semantic, and hybrid for K=5
for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type([gt_row_oncology_history], st, k=5)
    print(f"{st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

lexical: Hit@5 = 0.000, MRR@5 = 0.000
semantic: Hit@5 = 0.000, MRR@5 = 0.000
hybrid: Hit@5 = 0.000, MRR@5 = 0.000


All zeros mean that, for this specific question and K=5, none of the top‑5 retrieved chunks (for lexical, semantic, or hybrid) contained any chunks, so Hit@5 and MRR@5 both evaluate to 0.

Getting zeros at first is normal — it tells you:
 - this query is hard for your current retrieval setup,
 - it’s a good candidate for tuning:
    - adjust boost_dict in search() (e.g., give more weight to heading or doc_type),
    - try different rrf_k in hybrid_search(),
    - or experiment with filtering on doc_type or is_oncology for this question.

    

1.  First check what your search functions are actually returning. Do the two gold chunk IDs appear at all? If they only appear beyond rank 5, Hit@5 and MRR@5 will be 0, even though retrieval works somewhat.

In [24]:
question = gt_row_oncology_history["question"]
patient_id = gt_row_oncology_history["patient_id"]
gold_ids = set(gt_row_oncology_history["gold_chunk_ids"])

for st in ["lexical", "semantic", "hybrid"]:
    results = rag.search(question, patient_id=patient_id, num_results=10) if st == "lexical" else \
              rag.semantic_search(question, patient_id=patient_id, num_results=10) if st == "semantic" else \
              rag.hybrid_search(question, patient_id=patient_id, num_results=10)

    print(f"\n{st.upper()} top 10 chunk_ids:")
    for rank, doc in enumerate(results, start=1):
        cid = doc["chunk_id"]
        mark = " <-- GOLD" if cid in gold_ids else ""
        print(f"{rank}: {cid}{mark}")


LEXICAL top 10 chunk_ids:
1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
2: 6de9ad49acc6eaf72f6f9490837fd3cf1482f5d4
3: ded7f386dd5af59ce715fa6ca0a5cd8df1359b17
4: 0b13f12069a342ed9a671de226d1378fabeed089
5: 0eb007dc650d4e5da238ea21a804db7ac6284c71
6: 2649097cd5f3df6c62f85b9fc9d8331057c57827
7: 0590211551c10b6259f5a9062003667e93e8f4e6
8: 337e5a497cfd365de578eb2d70a9a8e5ce1b1be8
9: bd3f64eab5732a617f775caca585388e20b7ba2c
10: a7af7e0af688eaa2d25ae618de85e72d7be0e98c

SEMANTIC top 10 chunk_ids:
1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
2: 9317205de02c6aea7e12dc5d3b66c1f1542acfb1
3: 8e56262bd19ec431b869e7f7c27325ff99ecb3bf
4: 2649097cd5f3df6c62f85b9fc9d8331057c57827
5: 55253b39fc3a7cd3bbd91cd08ec1c21d076d5fc8
6: 9f458a6e73124021d5f0a899087e67059ddae149
7: 4fcfe36428998e8c919d8f5caca637716db257f0
8: 791b3dbd2f8a11a10fa2fbdf07869dc49f62c068
9: 250dc0cc0885b6032e6801bc4ef4bc8bcf45c81a
10: a4cfc966af3919383791ed67e603839a7d41411f

HYBRID top 10 chunk_ids:
1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38

2. Try evaluating at a larger K (e.g., 10 or 20).

If Hit@10 > 0 and MRR@10 > 0, that tells you:
- the retriever can find the oncology chunks, but not in the top 5;
- you may need to:
    - increase K during retrieval,
    - or tune lexical/semantic parameters so those chunks rise in rank

In [25]:
for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type([gt_row_oncology_history], st, k=20)
    print(f"{st}: Hit@20 = {hr:.3f}, MRR@20 = {mrr:.3f}")

lexical: Hit@20 = 1.000, MRR@20 = 0.077
semantic: Hit@20 = 0.000, MRR@20 = 0.000
hybrid: Hit@20 = 0.000, MRR@20 = 0.000


So now we see that for this question, lexical search eventually finds a gold chunk but only after rank 13, so Hit@5/MRR@5 are 0, and Hit@20=1, MRR@20≈1/13≈0.077.

Semantic and hybrid search never retrieve either gold chunk in the top 20 for this query, so their Hit@K and MRR@K stay 0.

That means the evaluation code is working, and your retrieval pipeline is behaving like this:
- lexical: weak but not completely failing,
    - Hit@5 = 0: no gold chunk among the top 5 → the LLM is unlikely to see the oncology evidence if you only feed 5 chunks into the prompt.
    - Hit@20 = 1: at least one gold chunk appears within the top 20.
    - MRR@20 ≈ 0.077: first relevant chunk is around rank 13, since 1/13≈0.077.
- semantic: not finding oncology-history chunks for this query,
- hybrid: dominated by whatever lexical/semantic return, so it also misses the gold chunks.

In [35]:
#you should see the rank around 13
gold_ids = set(gt_row_oncology_history["gold_chunk_ids"])

lex_results_20 = rag.search(
    query=gt_row_oncology_history["question"],
    patient_id=gt_row_oncology_history["patient_id"],
    num_results=20,
)

for rank, doc in enumerate(lex_results_20, start=1):
    cid = doc["chunk_id"]
    mark = " <-- GOLD" if cid in gold_ids else ""
    print(f"{rank}: {cid}{mark}")

1: e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
2: 6de9ad49acc6eaf72f6f9490837fd3cf1482f5d4
3: ded7f386dd5af59ce715fa6ca0a5cd8df1359b17
4: 0b13f12069a342ed9a671de226d1378fabeed089
5: 0eb007dc650d4e5da238ea21a804db7ac6284c71
6: 2649097cd5f3df6c62f85b9fc9d8331057c57827
7: 0590211551c10b6259f5a9062003667e93e8f4e6
8: 337e5a497cfd365de578eb2d70a9a8e5ce1b1be8
9: bd3f64eab5732a617f775caca585388e20b7ba2c
10: a7af7e0af688eaa2d25ae618de85e72d7be0e98c
11: f3d187ee2691f8cb0658ba7311e0a7c03f373cf5
12: dacdef41d3d429414bde9c1cf563a3c3158fba4a
13: 4412a94aa2cefe467b43d2f7588722ab03167c6e <-- GOLD
14: 4fcfe36428998e8c919d8f5caca637716db257f0
15: 392065e296a5586b78eb9e2a43d08df5220cb956
16: 9c78177c42bf66d596a1bfa5b0e361e039d1202b
17: 8507524fabeea1b5637f597d251bd4f162cf5d3a
18: 8123905ce9ef5c2a100a0400f86029469183dbb2
19: 0a66dc2602fa0cf65cf38785b2cbf0278b37bc76
20: 0a0c014c77255e8a09d093757d4d448cb0ee95fa


Hybrid search uses RRF over the top 10 lexical + top 10 semantic results. If the gold chunks are not in either list of 10, RRF can’t bring them up.

You’re embedding a text built as:
```
parts = [
    f"title: {doc['title']}",
    f"heading: {doc['heading']}",
    f"doc_type: {doc['doc_type']}",
    f"chunk_text: {doc['chunk_text']}",
]
```
and your question is:
“Summarize the patient’s oncology history.”

The `parts` variable corresponds to a `doc` (from `docs_for_patient = [d for d in rag.vector_documents if d["patient_id"] == patient_id]
`), see the cell below.
```
Number of docs for this patient: 1641

--- Doc 1 ---
title: oncology_timeline_events.csv
heading: oncology_timeline_events
doc_type: oncology_timeline_events
chunk_text: event_type: Condition; date: 1982-04-08T21:48:26-05:00; label: Acute myeloid leukemia, disease (disorder); status: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json

--- Doc 2 ---
...
```


The model doesn’t see a strong semantic match between this generic question and specific oncology chunks, especially if the chunks don’t use phrases like “oncology history” but instead mention specific events, dates, and terms.

There might be other chunks (e.g., diagnosis summaries, overviews) whose text is semantically closer to that query, so they rank higher even if you didn’t mark them as gold.


Seeing the `parts` or the full `build_embedding_text(doc)` output will help you understand why semantic search might not be matching this question strongly; you can then decide whether to tweak the embedding text format (e.g., prepend an “Oncology history” label for certain doc types).


In [36]:
import rag

# Pick all docs for this patient
patient_id = "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"
docs_for_patient = [d for d in rag.vector_documents if d["patient_id"] == patient_id]

print("Number of docs for this patient:", len(docs_for_patient))

# For example, inspect the first 3 docs
for i, doc in enumerate(docs_for_patient[:3], start=1):
    print(f"\n--- Doc {i} ---")
    parts = [
        f"title: {doc['title']}",
        f"heading: {doc['heading']}",
        f"doc_type: {doc['doc_type']}",
        f"chunk_text: {doc['chunk_text']}",
    ]
    for p in parts:
        print(p)

Number of docs for this patient: 1641

--- Doc 1 ---
title: oncology_timeline_events.csv
heading: oncology_timeline_events
doc_type: oncology_timeline_events
chunk_text: event_type: Condition; date: 1982-04-08T21:48:26-05:00; label: Acute myeloid leukemia, disease (disorder); status: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json

--- Doc 2 ---
title: oncology_timeline_events.csv
heading: oncology_timeline_events
doc_type: oncology_timeline_events
chunk_text: event_type: Procedure; date: 1982-04-08T21:48:26-05:00; label: Chemotherapy (procedure); status: completed; resource_id: 7b0cc75d-9ab5-df30-d406-c88245ea1c55; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json

--- Doc 3 ---
title: oncology_timeline_events.csv
heading: oncology_timeline_events
doc_type: oncology_timeline_events
chunk_text: event_type: Procedure; date: 2021-03-

In [ ]:
# This shows you exactly what text you’re feeding into the embedder for each chunk.

gold_ids = [
    "4412a94aa2cefe467b43d2f7588722ab03167c6e",
    "d0c6336772cc8ee0cc79cc2628bdba3218017dc8",
]

for cid in gold_ids:
    matches = [d for d in rag.vector_documents if d["chunk_id"] == cid]
    if not matches:
        print(f"\nNo doc found for chunk_id={cid}")
        continue

    doc = matches[0]
    print(f"\n=== Gold chunk {cid} ===")
    parts = [
        f"title: {doc['title']}",
        f"heading: {doc['heading']}",
        f"doc_type: {doc['doc_type']}",
        f"chunk_text: {doc['chunk_text']}",
    ]
    for p in parts:
        print(p)


=== Gold chunk 4412a94aa2cefe467b43d2f7588722ab03167c6e ===
title: Oncology Timeline: Beth967 Cremin516
heading: Timeline
doc_type: oncology_timeline
chunk_text: - 1982-04-08T21:48:26-05:00 — Condition: Acute myeloid leukemia, disease (disorder); detail: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3
- 1982-04-08T21:48:26-05:00 — Procedure: Chemotherapy (procedure); detail: completed; resource_id: 7b0cc75d-9ab5-df30-d406-c88245ea1c55
- 2021-03-26T23:21:45-04:00 — Procedure: Biopsy of breast (procedure); detail: completed; resource_id: 6a30e873-498a-13d2-36d9-c9cccd842e61
- 2021-03-26T23:52:56-04:00 — Condition: Malignant neoplasm of breast (disorder); detail: active; resource_id: 42681656-7b91-38ac-3c81-eb7ef59e1686
- 2021-03-26T23:52:56-04:00 — Observation: Distant metastases.clinical [Class] Cancer; detail: M0 category (finding); resource_id: 765ade52-22d2-4f79-5f9f-8bff8a7e71e5
- 2021-03-26T23:52:56-04:00 — Observation: Size Tumor; resource_id: acfdf303-eac2-143f-2d1b-

In [38]:
# If you want to see the final concatenated string your embedder sees

def build_embedding_text(doc):
    parts = [
        f"title: {doc['title']}" if doc["title"] else "",
        f"heading: {doc['heading']}" if doc["heading"] else "",
        f"doc_type: {doc['doc_type']}" if doc["doc_type"] else "",
        f"chunk_text: {doc['chunk_text']}" if doc["chunk_text"] else "",
    ]
    return "\n".join(part for part in parts if part).strip()

# Example for one gold chunk
gold_doc = [d for d in rag.vector_documents if d["chunk_id"] == gold_ids[0]][0]
print(build_embedding_text(gold_doc))

title: Oncology Timeline: Beth967 Cremin516
heading: Timeline
doc_type: oncology_timeline
chunk_text: - 1982-04-08T21:48:26-05:00 — Condition: Acute myeloid leukemia, disease (disorder); detail: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3
- 1982-04-08T21:48:26-05:00 — Procedure: Chemotherapy (procedure); detail: completed; resource_id: 7b0cc75d-9ab5-df30-d406-c88245ea1c55
- 2021-03-26T23:21:45-04:00 — Procedure: Biopsy of breast (procedure); detail: completed; resource_id: 6a30e873-498a-13d2-36d9-c9cccd842e61
- 2021-03-26T23:52:56-04:00 — Condition: Malignant neoplasm of breast (disorder); detail: active; resource_id: 42681656-7b91-38ac-3c81-eb7ef59e1686
- 2021-03-26T23:52:56-04:00 — Observation: Distant metastases.clinical [Class] Cancer; detail: M0 category (finding); resource_id: 765ade52-22d2-4f79-5f9f-8bff8a7e71e5
- 2021-03-26T23:52:56-04:00 — Observation: Size Tumor; resource_id: acfdf303-eac2-143f-2d1b-21baa9dbe1f8
- 2021-03-26T23:52:56-04:00 — Observation: Prima

## Picking better gold chunks

You’re right that randomly picking 2 chunks is not ideal for this question. For “Summarize the patient’s oncology history,” your gold set should include the key oncology-history chunks, which will often mean several chunks from oncology_timeline plus possibly some from conditions (or other oncology-relevant doc types), but not literally all oncology-timeline chunks.

If you mark every oncology_timeline chunk as gold:
- Hit@K becomes easier to achieve (any oncology chunk counts), but
- your metric becomes less informative:
    - retrieving a minor detail chunk (e.g., “follow-up CT showed no progression”) gets the same credit as retrieving the main diagnosis/treatment summary chunk.
    - you also risk labelling noisy or marginally relevant chunks as gold.

**Ground truth works best when gold_chunk_ids are the chunks you’d actually want the LLM to see for this question**, not all chunks that happen to be in the same document.

**Better strategy: “principal” oncology-history chunks**

For this question: “Summarize the patient’s oncology history.”

Prefer to label:
 - 1–2 summary-style chunks from oncology_timeline that:
        - introduce the cancer type, stage, diagnosis date;
        - outline major treatments and key events.

- plus 1–2 diagnosis-focused chunks from conditions if they:
        - clearly describe the primary cancer diagnosis, staging, or important comorbid oncologic conditions.

So instead of 2 random chunks, pick the **3–5 chunks** that:
- you, as a human, would copy/paste into a **human-written oncology history summary**;
- cover **diagnosis** and **treatment** across time.

Those become your gold_chunk_ids.


So for `onc_timeline` chunks manually read a handful of these chunks and pick the ones that:
- summarize diagnosis (what cancer, when, stage),
- summarize major treatments (chemo/radiation/surgery and their timing),
- or give a high-level timeline.

Similarly, scan `conditions` chunks for the clearest diagnosis description.

In [39]:
import rag

patient_id = "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

docs_for_patient = [
    d for d in rag.vector_documents
    if d["patient_id"] == patient_id and int(d.get("is_oncology", 0)) == 1
]

# Focus on oncology_timeline and conditions
onc_timeline = [d for d in docs_for_patient if d["doc_type"] == "oncology_timeline"]
conditions = [d for d in docs_for_patient if d["doc_type"] == "conditions"]

print("Oncology timeline chunks:", len(onc_timeline))
print("Conditions chunks:", len(conditions))

# Inspect a few oncology timeline chunks
for i, doc in enumerate(onc_timeline[:10], start=1):
    print(f"\n--- Oncology timeline chunk {i} ---")
    print("chunk_id:", doc["chunk_id"])
    print("heading:", doc["heading"])
    print(doc["chunk_text"][:500], "...")

Oncology timeline chunks: 9
Conditions chunks: 2

--- Oncology timeline chunk 1 ---
chunk_id: e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
heading: Oncology Timeline: Beth967 Cremin516
- Patient ID: 41681ed6-efc5-94c0-1bc0-f60b34dbd31b
- Oncology-related dated events: 49 ...

--- Oncology timeline chunk 2 ---
chunk_id: 4412a94aa2cefe467b43d2f7588722ab03167c6e
heading: Timeline
- 1982-04-08T21:48:26-05:00 — Condition: Acute myeloid leukemia, disease (disorder); detail: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3
- 1982-04-08T21:48:26-05:00 — Procedure: Chemotherapy (procedure); detail: completed; resource_id: 7b0cc75d-9ab5-df30-d406-c88245ea1c55
- 2021-03-26T23:21:45-04:00 — Procedure: Biopsy of breast (procedure); detail: completed; resource_id: 6a30e873-498a-13d2-36d9-c9cccd842e61
- 2021-03-26T23:52:56-04:00 — Condition: Malignant neoplasm of breast ( ...

--- Oncology timeline chunk 3 ---
chunk_id: dacdef41d3d429414bde9c1cf563a3c3158fba4a
heading: Timeline
3e
- 2021-03-26

In [41]:
for i, doc in enumerate(conditions[:10], start=1):
    print(f"\n--- Conditions chunk {i} ---")
    print("chunk_id:", doc["chunk_id"])
    print("heading:", doc["heading"])
    print(doc["chunk_text"][:500], "...")


--- Conditions chunk 1 ---
chunk_id: d0c6336772cc8ee0cc79cc2628bdba3218017dc8
heading: conditions
patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd31b; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; encounter_reference: urn:uuid:30571c41-c572-f58b-2674-87f47144b2b4; code_system: http://snomed.info/sct; code: 91861009; display: Acute myeloid leukemia, disease (disorder); text: Acute myeloid leukemia, disease (disorder); clinical_status: resolved; verification_status: confirmed; onset_datetime: 19 ...

--- Conditions chunk 2 ---
chunk_id: d1ce3fd72756c7deeb385af432c9d7f00ae6bfef
heading: conditions
patient_id: 41681ed6-efc5-94c0-1bc0-f60b34dbd31b; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json; resource_id: 42681656-7b91-38ac-3c81-eb7ef59e1686; encounter_reference: urn:uuid:ae3886aa-d916-10f0-0e3f-ad6711e25e25; code_system: http://snomed.inf

## 11 relevant chunks

Given that all 11 chunks (9 `oncology_timeline`, 2 `conditions`) are clearly relevant, those are excellent candidates for your gold labels for oncology-history questions. The question now is how to use them in a way that makes your evaluation informative rather than trivial.

### 1. What your counts tell you

From your query:

- Total chunks for this patient: 1641  (see above `len(chunks_df)` is 1641)
- Oncology-flagged chunks: 116  (see above `len(chunks_df[chunks_df["is_oncology"] == 1])` is 116)
- Among those 116: (see above `onc_chunks_patient["doc_type"].value_counts()`)
  - 9 chunks with `doc_type == "oncology_timeline"`
  - 2 chunks with `doc_type == "conditions"`

And you’ve inspected those 11 and found them all “very relevant” to oncology history. That’s exactly what you want as a basis for ground truth: a small, focused set of chunks that really describe the cancer journey. [pmc.ncbi.nlm.nih](https://pmc.ncbi.nlm.nih.gov/articles/PMC7265796/)

### 2. How to turn these into gold_chunk_ids

You can pull these 11 chunk IDs directly:

```python
oncology_docs = chunks_df[chunks_df["is_oncology"] == 1].copy()

# Focus on doc_type oncology_timeline and conditions
onc_history_docs = oncology_docs[
    oncology_docs["doc_type"].isin(["oncology_timeline", "conditions"])
].copy()

print("Selected oncology history chunks:", len(onc_history_docs))
display(onc_history_docs[["doc_type", "heading", "chunk_id"]])

gold_chunk_ids_oncology_history = oncology_history_docs["chunk_id"].tolist()
```

Then define your ground-truth row as:

```python
gt_row_oncology_history = {
    "patient_id": patient_id,
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": gold_chunk_ids_oncology_history,
}
```

Now you’re saying:

- any of these 11 chunks are valid “gold nuggets” for an oncology-history summary. [linkedin](https://www.linkedin.com/posts/dan-bucureanu_here-are-the-easy-steps-to-perform-your-rag-activity-7341344334752985089-537X)

### 3. Trade-off: all 11 vs a smaller subset

Using all 11 as gold_chunk_ids is reasonable here because:

- they’re not a huge number,
- they all genuinely contribute to oncology history.

However:

- It slightly **loosens** the metric: Hit@K becomes easier to satisfy, because any of 11 IDs counts as a hit.
- To keep the metric discriminative, you could:
  - rank those 11 by importance and keep, say, the 5 most central ones (main diagnosis, main timeline summaries, major treatment events),
  - or keep all 11 for this question but use stricter, smaller gold sets for other questions (e.g., “What treatments has the patient received?” might have only a few treatment-specific chunks). [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

A pragmatic approach:

- For **high-level summary questions** like this, use all oncology-timeline + key conditions chunks as gold.
- For **more specific questions** (e.g., “What chemotherapy regimen?”), define a narrower gold set. [levshaazz.github](https://levshaazz.github.io/deep-learning-for-search-summer-2026/en/assignments/hw-rag/)

### 4. Why this is better than 2 random chunks

Compared to your earlier two random picks, this new gold set:

- aligns with the actual document structure:
  - uses `doc_type` to focus on timeline and condition documents,
- better reflects what a human would want to see when summarizing oncology history,
- gives your retrieval metrics a fairer goal: bring *any* of the clearly oncology history chunks into the top K. [tavrn](https://www.tavrn.ai/blog/medical-chronology-summary)

Once you set `gold_chunk_ids` to these 11 IDs, rerun Hit@K and MRR for lexical/semantic/hybrid. Even if Hit@5 is still low, you’ll know you’re evaluating against a more principled gold standard.

Would you like a notebook cell that automatically builds `gold_chunk_ids` for oncology-history questions for any patient by selecting `oncology_timeline` and `conditions` chunks with `is_oncology == 1`? NO!

## Test 11 chunks Hit@5 = 1.000, MRR@5 = 1.000

If you eventually get all 1s (Hit@K=1 and MRR@K=1 across your evaluation set), that’s *possible* and not logically inconsistent — it would just mean your retriever is doing an excellent job on the particular questions and gold labels you chose. It’s not “wrong,” but it is **unusual** for anything beyond a very small, curated test set.

A few nuances:

- For a **single question** with a good filter (e.g., only `oncology_timeline` + `conditions`) and a clear query, Hit@5=1 and MRR@5=1 are absolutely possible: the first retrieved chunk is one of your gold chunks, and you mark that as a perfect success. That’s fine. [aievals](https://www.aievals.co/learn/rag-evals/retrieval-metrics)

- For **many questions**:
  - if you consistently get Hit@K=1 and MRR@K=1, it usually means:
    - either your retrieval is extremely well tuned for that narrow corpus, **or**
    - your gold labeling is broad enough that almost any reasonable chunk counts as “correct.”  
  - In practice, most RAG systems see a mix: some questions with perfect scores, many with partial success, and some misses. [youtube](https://www.youtube.com/watch?v=9evIzHs9d60)

So:

- It’s not unexpected or “invalid” if you see all 1s on a small, easy set of questions where:
  - the gold chunks are obvious,
  - the query phrasing lines up well with the chunk text,
  - metadata filtering perfectly scopes the search. [youtube](https://www.youtube.com/watch?v=IYx4O42dHd0)

- It *would* be a red flag only if:
  - you have a wide variety of realistic, challenging questions, and
  - your metrics are still trivially perfect — that usually suggests the evaluation setup is too forgiving (e.g., gold sets that include nearly all chunks in the filtered subset, or questions that are nearly identical to chunk headings). [dataaihub](https://www.dataaihub.co/learn/retrieval-evaluation)

In your case, if for this one oncology-history question you eventually get Hit@5=1 and MRR@5=1 after tuning, that’s a good sign: it means your retrieval stack surfaces a clearly relevant oncology chunk as the very first result when searching the right “shelf.”

In [48]:
onc_docs = chunks_df[chunks_df["is_oncology"] == 1].copy()

# Focus on doc_type oncology_timeline and conditions
onc_history_docs = onc_docs[
    onc_docs["doc_type"].isin(["oncology_timeline", "conditions"])
].copy()

print("Selected oncology history chunks:", len(onc_history_docs))
display(onc_history_docs[["doc_type", "heading", "chunk_id"]])

gold_chunk_ids_onc_history = onc_history_docs["chunk_id"].tolist()

Selected oncology history chunks: 11


,doc_type,heading,chunk_id
6,conditions,conditions,d0c6336772cc8ee0cc79cc2628bdba3218017dc8
52,conditions,conditions,d1ce3fd72756c7deeb385af432c9d7f00ae6bfef
1109,oncology_timeline,Oncology Timeline: Beth967 Cremin516,e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862
1110,oncology_timeline,Timeline,4412a94aa2cefe467b43d2f7588722ab03167c6e
1111,oncology_timeline,Timeline,dacdef41d3d429414bde9c1cf563a3c3158fba4a
1112,oncology_timeline,Timeline,4fcfe36428998e8c919d8f5caca637716db257f0
1113,oncology_timeline,Timeline,bd3f64eab5732a617f775caca585388e20b7ba2c
1114,oncology_timeline,Timeline,a7af7e0af688eaa2d25ae618de85e72d7be0e98c
1115,oncology_timeline,Timeline,f3d187ee2691f8cb0658ba7311e0a7c03f373cf5
1116,oncology_timeline,Timeline,2649097cd5f3df6c62f85b9fc9d8331057c57827


In [49]:
gt_row_onc_history = {
    "patient_id": patient_id,
    "question": "Summarize the patient's oncology history.",
    "gold_chunk_ids": gold_chunk_ids_onc_history,
}

In [50]:
import math
import rag as rag

def get_ranked_chunk_ids(query, patient_id, search_type, k=5):
    """Run the chosen search and return top-k chunk_ids in rank order."""
    if search_type == "lexical":
        results = rag.search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "semantic":
        results = rag.semantic_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    elif search_type == "hybrid":
        results = rag.hybrid_search(
            query=query,
            patient_id=patient_id,
            num_results=k,
        )
    else:
        raise ValueError("search_type must be one of: lexical, semantic, hybrid")

    return [doc["chunk_id"] for doc in results]

def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """Hit@K: 1 if ANY relevant chunk appears in top K, else 0."""
    relevant = set(relevant_ids)
    top_k = retrieved_ids[:k]
    return int(any(doc_id in relevant for doc_id in top_k))

def mrr_at_k(retrieved_ids, relevant_ids, k):
    """MRR@K: 1/rank of first relevant chunk in top K, else 0."""
    relevant = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

def eval_search_type(gt_row_oncology_history, search_type, k=5):
    hits = []
    mrrs = []

    for row in gt_row_oncology_history:
        patient_id = row["patient_id"]
        question = row["question"]
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

    return hit_rate, mrr

# Evaluate lexical, semantic, and hybrid for K=5
for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type([gt_row_onc_history], st, k=5)
    print(f"{st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

lexical: Hit@5 = 1.000, MRR@5 = 1.000
semantic: Hit@5 = 1.000, MRR@5 = 1.000
hybrid: Hit@5 = 1.000, MRR@5 = 1.000


## If Hit@20 and MRR@20 were still zeros:

3. Confirm the gold IDs are correct for this index

They must be present in vector_index_metadata.json and in the minsearch_documents.json / metadata.db that rag.py loads for this environment.


In [26]:
import json
from pathlib import Path

vector_meta = json.loads(Path("../data/retrieval/vector_index_metadata.json").read_text())
docs = vector_meta["documents"]
ids_in_vector = {doc["chunk_id"] for doc in docs}

for cid in gold_ids:
    print(cid, cid in ids_in_vector)

4412a94aa2cefe467b43d2f7588722ab03167c6e True
d0c6336772cc8ee0cc79cc2628bdba3218017dc8 True


4. Confirm that search is returning anything at all


In [27]:
question = gt_row_oncology_history["question"]
patient_id = gt_row_oncology_history["patient_id"]

for st in ["lexical", "semantic", "hybrid"]:
    if st == "lexical":
        results = rag.search(
            query=question,
            patient_id=patient_id,
            num_results=10,
        )
    elif st == "semantic":
        results = rag.semantic_search(
            query=question,
            patient_id=patient_id,
            num_results=10,
        )
    else:
        results = rag.hybrid_search(
            query=question,
            patient_id=patient_id,
            num_results=10,
        )

    print(f"\n{st.upper()} — number of results:", len(results))
    for i, doc in enumerate(results, start=1):
        print(
            f"{i}: chunk_id={doc.get('chunk_id')}, "
            f"patient_id={doc.get('patient_id')}, "
            f"doc_type={doc.get('doc_type')}, "
            f"title={doc.get('title')}, "
            f"is_oncology={doc.get('is_oncology')}"
        )


LEXICAL — number of results: 10
1: chunk_id=e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=oncology_timeline, title=Oncology Timeline: Beth967 Cremin516, is_oncology=1
2: chunk_id=6de9ad49acc6eaf72f6f9490837fd3cf1482f5d4, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=patient_overview, title=Patient Overview: Beth967 Cremin516, is_oncology=1
3: chunk_id=ded7f386dd5af59ce715fa6ca0a5cd8df1359b17, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=patient_overview, title=Patient Overview: Beth967 Cremin516, is_oncology=1
4: chunk_id=0b13f12069a342ed9a671de226d1378fabeed089, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=oncology_timeline, title=Oncology Timeline: Beth967 Cremin516, is_oncology=1
5: chunk_id=0eb007dc650d4e5da238ea21a804db7ac6284c71, patient_id=41681ed6-efc5-94c0-1bc0-f60b34dbd31b, doc_type=patient_overview, title=Patient Overview: Beth967 Cremin516, is_oncology=1
6: chunk_id=2649097cd5f3df6c

5. Check if the indices were built from a different manifest. It may be that:
- the current metadata.db / vector_index_metadata.json do not contain any chunks for this patient_id; or
- they contain chunks, but your rag module in the notebook is loading an older index file.

In [28]:
# Check lexical documents (if you have minsearch_documents.json)
from pathlib import Path
import json

docs_path = Path("../data/retrieval/minsearch_documents.json")
if docs_path.exists():
    min_docs = json.loads(docs_path.read_text())
    patient_ids_lexical = {doc["patient_id"] for doc in min_docs}
    print("Patient in lexical docs:", patient_id in patient_ids_lexical)

# Check semantic documents (vector index metadata)
vec_meta_path = Path("../data/retrieval/vector_index_metadata.json")
vec_meta = json.loads(vec_meta_path.read_text())
vec_docs = vec_meta["documents"]
patient_ids_vector = {doc["patient_id"] for doc in vec_docs}
print("Patient in vector docs:", patient_id in patient_ids_vector)

Patient in lexical docs: True
Patient in vector docs: True


6. Verify that rag in your notebook is loading the right files

In [29]:
import rag
from pathlib import Path

print("Lexical index path:", rag.INDEX_PATH, "exists:", Path(rag.INDEX_PATH).exists())
print("Vector index path:", rag.VECTOR_INDEX_PATH, "exists:", Path(rag.VECTOR_INDEX_PATH).exists())
print("Vector metadata path:", rag.VECTOR_METADATA_PATH, "exists:", Path(rag.VECTOR_METADATA_PATH).exists())

Lexical index path: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval/minsearch_index.pkl exists: True
Vector index path: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval/vector_index.npz exists: True
Vector metadata path: /Users/barbarato/llm-zoomcamp-capstone-project/data/retrieval/vector_index_metadata.json exists: True


If all three search modes return zero results for this patient and question, even though the patient’s chunks are in the indices, then some filter condition is excluding every document. In your code, the only filters are on patient_id, doc_type, and is_oncology, so either:
- patient_id in the docs does not exactly match the string you’re passing, or
- for lexical search, minsearch’s filter_dict filtering is stricter than expected, causing it to drop everything.

In [30]:
import rag

# Check lexical docs (minsearch documents)
lex_docs = rag.index.docs  # or rag.index.docs depending on minsearch version
print("Total lexical docs:", len(lex_docs))

# Filter docs by patient id as stored in the index
lex_patient_docs = [d for d in lex_docs if d.get("patient_id") == "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"]
print("Lexical docs for this patient:", len(lex_patient_docs))
print(lex_patient_docs[:3])

Total lexical docs: 117021
Lexical docs for this patient: 1641
[{'id': '0458f43f3038f41f95741728d84d2afb01297836', 'chunk_id': '0458f43f3038f41f95741728d84d2afb01297836', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163e0abeac8d66324a6969f55b32afd7028c8', 'doc_type': 'oncology_timeline_events', 'title': 'oncology_timeline_events.csv', 'heading': 'oncology_timeline_events', 'chunk_text': 'event_type: Condition; date: 1982-04-08T21:48:26-05:00; label: Acute myeloid leukemia, disease (disorder); status: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json', 'chunk_index': 0, 'is_oncology': '1', 'date_start': '1982-04-08T21:48:26-05:00', 'date_end': '1982-04-08T21:48:26-05:00'}, {'id': 'bcc330d0e3e803819caec7ebc2cc3e5f6f4f6d9c', 'chunk_id': 'bcc330d0e3e803819caec7ebc2cc3e5f6f4f6d9c', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163

If len(lex_patient_docs) is 0 here (even though Patient in lexical docs: True from the JSON file check), it means the patient_id values actually stored in the minsearch index differ from the string you’re using (e.g., whitespace, different casing, or a different ID altogether).

Do a quick “what patient IDs exist?”:

In [31]:
patient_ids_in_index = {d.get("patient_id") for d in lex_docs}
print("Some patient_ids in index:", list(patient_ids_in_index)[:10])
# Compare this to "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

Some patient_ids in index: ['ee23ebc7-cc3c-862c-03bc-46e9321c0df1', '6e424da5-2702-49de-4046-868580fe7235', '25197dc8-9425-1999-5914-f2171b0d4e32', '8629e803-6f0f-d985-e314-ff0808ac8bd4', '43a372f7-4803-75a8-9f34-45f6d377589f', 'deea7c91-1fca-a3a2-6c68-c7acced25766', '6fb374e8-33aa-a5ea-f050-b61394dfcb99', 'e595cd98-c5d0-2a9d-ea2e-a1dc325005fa', '64ae3769-65e4-222e-6793-1a3bc14ec682', '397b2de6-ccd8-858f-bf4a-b6fc379589bd']


In [32]:
vec_docs = rag.vector_documents
vec_patient_docs = [d for d in vec_docs if d.get("patient_id") == "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"]
print("Vector docs for this patient:", len(vec_patient_docs))
print(vec_patient_docs[:3])

Vector docs for this patient: 1641
[{'chunk_id': '0458f43f3038f41f95741728d84d2afb01297836', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163e0abeac8d66324a6969f55b32afd7028c8', 'doc_type': 'oncology_timeline_events', 'title': 'oncology_timeline_events.csv', 'heading': 'oncology_timeline_events', 'chunk_text': 'event_type: Condition; date: 1982-04-08T21:48:26-05:00; label: Acute myeloid leukemia, disease (disorder); status: resolved; resource_id: f5816124-5c61-9c37-04f1-343d390b6be3; source_file: data/prototype/sample50/Beth967_Cremin516_41681ed6-efc5-94c0-1bc0-f60b34dbd31b.json', 'chunk_index': 0, 'is_oncology': 1, 'date_start': '1982-04-08T21:48:26-05:00', 'date_end': '1982-04-08T21:48:26-05:00'}, {'chunk_id': 'bcc330d0e3e803819caec7ebc2cc3e5f6f4f6d9c', 'patient_id': '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'document_id': '209163e0abeac8d66324a6969f55b32afd7028c8', 'doc_type': 'oncology_timeline_events', 'title': 'oncology_timeline_events.csv', 'heading': '

In [33]:
patient_ids_in_index = {d.get("patient_id") for d in vec_docs}
print("Some patient_ids in index:", list(patient_ids_in_index)[:10])
# Compare this to "41681ed6-efc5-94c0-1bc0-f60b34dbd31b"

Some patient_ids in index: ['ee23ebc7-cc3c-862c-03bc-46e9321c0df1', '6e424da5-2702-49de-4046-868580fe7235', '25197dc8-9425-1999-5914-f2171b0d4e32', '8629e803-6f0f-d985-e314-ff0808ac8bd4', '43a372f7-4803-75a8-9f34-45f6d377589f', 'deea7c91-1fca-a3a2-6c68-c7acced25766', '6fb374e8-33aa-a5ea-f050-b61394dfcb99', 'e595cd98-c5d0-2a9d-ea2e-a1dc325005fa', '64ae3769-65e4-222e-6793-1a3bc14ec682', '397b2de6-ccd8-858f-bf4a-b6fc379589bd']


2. Test search without the patient filter to see whether query + index works at all.

If you get non-zero results here, then:
- your retrieval logic works,
- and the patient filter is the thing eliminating everything.


In [34]:
question = gt_row_oncology_history["question"]

# Lexical without patient filter
lex_results_no_filter = rag.search(
    query=question,
    patient_id=None,   # Force no filter
    num_results=10,
)
print("LEXICAL (no patient filter), n:", len(lex_results_no_filter))
print([r["patient_id"] for r in lex_results_no_filter])

# Semantic without patient filter
sem_results_no_filter = rag.semantic_search(
    query=question,
    patient_id=None,
    num_results=10,
)
print("SEMANTIC (no patient filter), n:", len(sem_results_no_filter))
print([r["patient_id"] for r in sem_results_no_filter])

LEXICAL (no patient filter), n: 10
['66e681dd-d945-9aba-95cb-d9606594cc9c', 'db62bf97-9977-e94c-94d1-46fa7abc524f', 'f18bd61f-f7be-ef83-db73-bbe83fb3391c', 'd44ac721-88ff-81cc-dd2f-7a30a1c1a221', '66d5d38f-4e52-3896-822a-541e29cd965b', 'cf018f86-f6b8-12a1-9f4d-ba14ff081fb4', '940b81eb-27fb-52d3-251e-df6edb18d893', '3af995f1-02a5-07ee-5a7e-e2470a017f1e', '8629e803-6f0f-d985-e314-ff0808ac8bd4', '8dfff5d0-a8f1-4eac-3987-1670cc41239b']
SEMANTIC (no patient filter), n: 10
['b73cbc40-ef87-47d5-08dc-4288048dbf1c', 'b31cb2e6-f4db-0f20-43e7-6ab691c2a32b', '0c0f2095-e8ab-7ac4-6ef4-625748255480', 'e595cd98-c5d0-2a9d-ea2e-a1dc325005fa', 'b9b9379e-68a1-49a5-4292-5cc3380e30d2', '64ae3769-65e4-222e-6793-1a3bc14ec682', 'b9b9379e-68a1-49a5-4292-5cc3380e30d2', '8dfff5d0-a8f1-4eac-3987-1670cc41239b', 'c457bf0f-0a34-2ebf-cec6-a385fb33d8a8', '0f5704ee-b38b-5a68-449d-9c44806517d0']


# FILTERING

I asked about the role of doc-type in my RAG:

Intuition: what doc_type lets you do in practice
Because of this wiring, in your notebook you can write queries like:

“Summarize the patient’s oncology history”
→ call lexical/semantic/hybrid with doc_types=["oncology_timeline", "conditions"].

“What medications is the patient taking?”
→ restrict to doc_types=["medications"].

“List all diagnostic reports”
→ doc_types=["diagnostic_reports"].

Each query hits only the relevant “shelf” in your clinical record library, rather than scanning every chunk. This is exactly the kind of metadata-based document-type filtering production RAG systems use to keep retrieval precise and efficient.

How to interpret your metrics with filters
Given your setup:

Pre-filtering by doc_type, patient_id, is_oncology ensures:

you’re evaluating retrieval within the right clinical scope,

and Hit/MRR measure ranking quality inside that scope.

Non-perfect metrics tell you:

how well your ranking (lexical/semantic/hybrid) surfaces key oncology chunks among other oncology chunks,

whether tuning (boosts, rrf_k, embedding text) improves that ranking.

So it’s expected — and actually useful — that Hit@5 and MRR@5 aren’t 1.0, even when metadata filtering is correctly scoping the search to oncology documents. The filters just ensure you’re grading the retriever on the right shelf, but the retriever still has to choose the right books.

## Selecting questions
<a id='selecting_questions'></a>

You don’t have to (and generally shouldn’t) use the *same exact question* for every remaining patient, but it **is** useful to have some repeated question patterns across patients.

A good setup is:

***

1. Reuse patterns, not necessarily exact wording

For your 9 patients, define questions that follow a few **shared templates**, but adapt them to what’s actually present in each record:

- Oncology patients:
  - “Summarize the patient’s oncology history.”
  - “What cancer treatments has the patient received?”
- All patients:
  - “What chronic conditions are documented for this patient?”
  - “What medications is the patient currently taking?”
  - “What major procedures has the patient undergone?”

You can reuse the exact wording across patients if it makes sense (e.g., all have `conditions` and `medications`), but if a patient has no oncology data, you’d skip the oncology-history question for that patient or explicitly treat it as a negative case (gold_chunk_ids = []). [devbook](https://devbook.zip/ai--and--ml/llm/context-engineering/rag/evaluation/retrieval-evaluation-sets)

***

2. Why having repeated patterns is helpful

- It lets you see how retrieval performs for **the same type of question across different patients**, which is useful for tuning.
- It keeps labeling effort manageable: you’re not inventing totally new question types for each patient, just reusing a small set of clinically meaningful templates. [youtube](https://www.youtube.com/watch?v=ufRxM1dp76Q)

So:

- Yes, you can use “Summarize the patient’s oncology history.” for *each patient who actually has oncology data*, with appropriate gold chunks.
- For patients without oncology, use other question types (conditions, medications, procedures) instead.

In [Finding gold chunks] section at the top we have:
      - For **9 patients**:
        - If you do 3 questions each → ~27 questions.
        - If you do 4 each → ~36 questions.

      That’s usually enough to:

      - get reasonably stable Hit@K and MRR estimates,
      - compare lexical vs semantic vs hybrid search,
      - and tune a few parameters (boosts, `rrf_k`, K) without your labeling workload exploding. [dataaihub](https://www.dataaihub.co/learn/retrieval-evaluation)

      Try to ensure your question set:

      - covers different document types (`patient_overview`, `oncology_timeline`, `conditions`, `medications`, etc.),
      - includes both easy and harder questions (some with answers in one chunk, some needing multiple chunks),
      - includes both oncology and non-oncology questions so the `is_oncology` metadata actually matters


So before designing the question set, it makes sense to inspect the doc_type coverage per patient.

In [60]:
import pandas as pd

manifest_path = project_root / "data" / "processed" / "mcode_breast_sample_50_manifest.csv"

# Load manifest
df = pd.read_csv(manifest_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (50, 17)

Columns:
['filename', 'patient_id', 'patient_name', 'n_resources', 'n_encounters', 'n_observations', 'n_conditions', 'n_procedures', 'n_medication_requests', 'n_medication_administrations', 'n_diagnostic_reports', 'first_date', 'last_date', 'followup_days', 'complexity_score', 'complexity_bucket', 'sample_seed']


In [61]:
# Check bucket distribution
print("Bucket counts in full manifest:")
print(df["complexity_bucket"].value_counts())

# Sample 3 patients from each bucket (low, medium, high)
bucket_targets = {"low": 3, "medium": 3, "high": 3}
selected_rows = []

for bucket, n in bucket_targets.items():
    bucket_df = df[df["complexity_bucket"] == bucket].copy()
    if len(bucket_df) < n:
        raise ValueError(f"Not enough patients in bucket '{bucket}' to sample {n}.")

    # Random sample with a fixed seed for reproducibility
    sampled_bucket = bucket_df.sample(n=n, random_state=42)
    selected_rows.append(sampled_bucket)

selected_df = pd.concat(selected_rows).reset_index(drop=True)

print("\nSelected 9 patients (3 per bucket):")
display(selected_df[["patient_id", "patient_name", "complexity_bucket",
                     "n_resources", "complexity_score"]])

# Just the list of patient_ids for later use
selected_patient_ids = selected_df["patient_id"].tolist()
print("\nSelected patient_ids:", selected_patient_ids)

Bucket counts in full manifest:
complexity_bucket
low       17
high      17
medium    16
Name: count, dtype: int64

Selected 9 patients (3 per bucket):


,patient_id,patient_name,complexity_bucket,n_resources,complexity_score
0,d65197b3-056a-2136-b584-77f43c29da3f,Corrie32 Boyle917,low,230,317
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,Florine959 Stark857,low,261,330
2,4736727e-63f4-071a-1516-a49310f5a052,Mónica985 Serrato62,low,436,602
3,29f6beee-162f-0113-7884-72245814693f,Eula461 Crooks415,medium,1854,2786
4,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,Beth967 Cremin516,medium,1843,2800
5,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,Deeann517 Torp761,medium,2191,3340
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,Francina926 Von197,high,2945,4458
7,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,Rosetta750 Stroman228,high,3055,4536
8,f203e11d-5573-1624-69b8-af8436987b3e,Shawana711 Lakin515,high,3365,4812



Selected patient_ids: ['d65197b3-056a-2136-b584-77f43c29da3f', 'f3739580-797d-ae04-eebf-aeddb2fc2f64', '4736727e-63f4-071a-1516-a49310f5a052', '29f6beee-162f-0113-7884-72245814693f', '41681ed6-efc5-94c0-1bc0-f60b34dbd31b', 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494', 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927', '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678', 'f203e11d-5573-1624-69b8-af8436987b3e']


In [62]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [64]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display

db_path = Path("../data/retrieval/metadata.db")
conn = sqlite3.connect(db_path)

# Build a parameterized IN clause for SQLite
placeholders = ",".join(["?"] * len(selected_patient_ids))

doc_type_counts_selected = pd.read_sql_query(
    f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        COUNT(*) AS n_chunks,
        SUM(CASE WHEN chunks.is_oncology = 1 THEN 1 ELSE 0 END) AS n_oncology_chunks
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    GROUP BY chunks.patient_id, documents.doc_type
    ORDER BY chunks.patient_id, n_chunks DESC
    """,
    conn,
    params=selected_patient_ids,
)

conn.close()

print("Long format: one row per selected patient/doc_type")
display(doc_type_counts_selected)

# Wide pivot: rows = selected patients, columns = doc_types, values = total chunk counts
doc_type_pivot_selected = (
    doc_type_counts_selected
    .pivot(index="patient_id", columns="doc_type", values="n_chunks")
    .fillna(0)
    .astype(int)
)

print("\nChunk counts by doc_type for each selected patient")
display(doc_type_pivot_selected)

# Oncology-only pivot
oncology_pivot_selected = (
    doc_type_counts_selected
    .pivot(index="patient_id", columns="doc_type", values="n_oncology_chunks")
    .fillna(0)
    .astype(int)
)

print("\nOncology-flagged chunk counts by doc_type for each selected patient")
display(oncology_pivot_selected)

# List of doc_types present per selected patient
doc_types_per_selected_patient = (
    doc_type_counts_selected[doc_type_counts_selected["n_chunks"] > 0]
    .groupby("patient_id")["doc_type"]
    .apply(list)
    .reset_index(name="doc_types")
)

print("\nDoc types present for each selected patient")
display(doc_types_per_selected_patient)

Long format: one row per selected patient/doc_type


,patient_id,doc_type,n_chunks,n_oncology_chunks
0,29f6beee-162f-0113-7884-72245814693f,observations,571,14
1,29f6beee-162f-0113-7884-72245814693f,procedures,408,37
2,29f6beee-162f-0113-7884-72245814693f,diagnostic_reports,284,0
3,29f6beee-162f-0113-7884-72245814693f,encounters,231,0
4,29f6beee-162f-0113-7884-72245814693f,oncology_timeline_events,53,53
...,...,...,...,...
76,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline_events,22,22
77,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,11,0
78,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,9,9
79,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline,5,5



Chunk counts by doc_type for each selected patient


doc_type,conditions,diagnostic_reports,encounters,medications,observations,oncology_timeline,oncology_timeline_events,patient_overview,procedures
patient_id,,,,,,,,,
29f6beee-162f-0113-7884-72245814693f,42,284,231,17,571,9,53,9,408
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,95,417,273,119,1133,6,26,9,651
41681ed6-efc5-94c0-1bc0-f60b34dbd31b,56,250,197,41,565,9,49,9,465
4736727e-63f4-071a-1516-a49310f5a052,6,62,61,8,151,9,47,9,54
aee216e6-cbe8-eaf2-3241-4bd1e8a01494,50,336,287,45,597,9,52,9,500
d65197b3-056a-2136-b584-77f43c29da3f,2,28,27,20,80,6,29,9,27
ecc4a7d0-8838-36b4-44ba-676d5a1f7927,115,394,262,72,970,9,48,9,760
f203e11d-5573-1624-69b8-af8436987b3e,95,404,270,40,1322,9,52,9,791
f3739580-797d-ae04-eebf-aeddb2fc2f64,5,26,25,11,119,5,22,9,23



Oncology-flagged chunk counts by doc_type for each selected patient


doc_type,conditions,diagnostic_reports,encounters,medications,observations,oncology_timeline,oncology_timeline_events,patient_overview,procedures
patient_id,,,,,,,,,
29f6beee-162f-0113-7884-72245814693f,2,0,0,0,14,9,53,9,37
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,2,0,0,0,14,6,26,0,10
41681ed6-efc5-94c0-1bc0-f60b34dbd31b,2,0,0,0,11,9,49,9,36
4736727e-63f4-071a-1516-a49310f5a052,1,0,0,0,11,9,47,9,35
aee216e6-cbe8-eaf2-3241-4bd1e8a01494,2,0,0,0,14,9,52,9,36
d65197b3-056a-2136-b584-77f43c29da3f,1,0,0,0,11,6,29,9,17
ecc4a7d0-8838-36b4-44ba-676d5a1f7927,2,0,0,0,10,9,48,9,36
f203e11d-5573-1624-69b8-af8436987b3e,2,0,0,0,14,9,52,0,36
f3739580-797d-ae04-eebf-aeddb2fc2f64,2,0,0,0,10,5,22,9,10



Doc types present for each selected patient


,patient_id,doc_types
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,..."
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,..."
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,..."
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,..."


In [65]:
doc_types_per_selected_patient

,patient_id,doc_types
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,..."
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,..."
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,..."
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,..."


In [67]:
import pandas as pd
from IPython.display import display

# Example: doc_types_per_selected_patient already computed, like:
# doc_types_per_selected_patient = (
#     doc_type_counts_selected[doc_type_counts_selected["n_chunks"] > 0]
#     .groupby("patient_id")["doc_type"]
#     .apply(list)
#     .reset_index(name="doc_types")
# )

# print("Doc types per selected patient:")
# display(doc_types_per_selected_patient)

# Convert each list of doc_types into a frozenset so order doesn't matter
doc_types_per_selected_patient["doc_types_set"] = (
    doc_types_per_selected_patient["doc_types"]
    .apply(lambda lst: frozenset(lst))
)

# Use the first patient's set as reference
reference_set = doc_types_per_selected_patient["doc_types_set"].iloc[0]
print("\nReference doc_types_set:", reference_set)

# Check equality against the reference
doc_types_per_selected_patient["matches_reference"] = (
    doc_types_per_selected_patient["doc_types_set"] == reference_set
)

print("\nEquality check vs reference:")
display(doc_types_per_selected_patient[["patient_id", "doc_types", "matches_reference"]])

# Summarize: how many distinct doc_types sets exist?
unique_sets = doc_types_per_selected_patient["doc_types_set"].unique()
print("\nNumber of distinct doc_types sets among selected patients:", len(unique_sets))

if len(unique_sets) > 1:
    print("Patients grouped by their doc_types_set:")
    # For readability, show each unique set and which patients have it
    for s in unique_sets:
        patients_with_set = doc_types_per_selected_patient[
            doc_types_per_selected_patient["doc_types_set"] == s
        ]["patient_id"].tolist()
        print(f"\nSet: {sorted(list(s))}")
        print("Patients:", patients_with_set)


Reference doc_types_set: frozenset({'oncology_timeline', 'encounters', 'diagnostic_reports', 'observations', 'patient_overview', 'oncology_timeline_events', 'procedures', 'medications', 'conditions'})

Equality check vs reference:


,patient_id,doc_types,matches_reference
0,29f6beee-162f-0113-7884-72245814693f,"[observations, procedures, diagnostic_reports,...",True
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,"[observations, procedures, diagnostic_reports,...",True
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,"[observations, procedures, diagnostic_reports,...",True
3,4736727e-63f4-071a-1516-a49310f5a052,"[observations, diagnostic_reports, encounters,...",True
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,"[observations, procedures, diagnostic_reports,...",True
5,d65197b3-056a-2136-b584-77f43c29da3f,"[observations, oncology_timeline_events, diagn...",True
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,"[observations, procedures, diagnostic_reports,...",True
7,f203e11d-5573-1624-69b8-af8436987b3e,"[observations, procedures, diagnostic_reports,...",True
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,"[observations, diagnostic_reports, encounters,...",True



Number of distinct doc_types sets among selected patients: 1


## Shared set of question

That’s a very good sign: it means all 9 selected patients have the **same set of document types** (even if the order in the lists differs).

Practically, this gives you a clean foundation for your evaluation set:

- Every selected patient has all of:
  - `patient_overview`
  - `conditions`
  - `medications`
  - `procedures`
  - `observations`
  - `diagnostic_reports`
  - `encounters`
  - `oncology_timeline_events`
  - `oncology_timeline` (plus whatever else is in that set).

So you can safely design a **shared set of question templates** and apply them across all 9 patients, only skipping oncology-specific ones for those who truly have no oncology-flagged content. [youtube](https://www.youtube.com/watch?v=IYx4O42dHd0)

A natural next step, given this:

- Define 3–4 question types that map onto these doc_types (e.g., overview, conditions, medications, oncology history).
- Then, for each of your 9 patients, label gold_chunk_ids for those questions using the doc_type + is_oncology filters you already have.

If you want, I can propose a concrete list of 4 question templates that directly correspond to these doc_types and are suitable for all 9 patients.

Great — here’s a concrete 4-question core set plus a 5-question extended set, both designed to work across all 9 selected patients, along with how to choose `gold_chunk_ids` for each. Because all patients share the same doc_types, you can apply this uniformly. 

### Core 4-question set (recommended baseline)

Use this for your main evaluation; it balances general and oncology content and stays relatively easy to annotate.

1. **Patient overview**

   - Question:  
     “Give a concise overview of this patient’s medical background and current care context.”
   - Primary doc_types to draw gold chunks from:  
     `patient_overview`, `encounters`, `conditions`, `oncology_timeline_events`. 
   - Gold chunk guidance:
     - Include 2–4 chunks that together cover:
       - Key chronic conditions. 
       - Major past procedures or events if they define the patient’s course. 
       - A high-level oncology context if applicable (e.g., diagnosis, line of therapy). 
     - Prefer chunks that are explicitly summary-like (e.g., overview notes) over raw measurements.

2. **Conditions**

   - Question:  
     “What are the patient’s main diagnosed conditions?”
   - Primary doc_types:  
     `conditions`, possibly `encounters` or `patient_overview` if they contain canonical lists. 
   - Gold chunk guidance:
     - Select chunks that list named diagnoses (problem lists, diagnosis sections, structured condition records). 
     - If conditions evolve (e.g., disease progression or resolved conditions), include chunks that clearly mark the current status.

3. **Medications**

   - Question:  
     “What medications is the patient taking or has recently taken?”
   - Primary doc_types:  
     `medications`, optionally `encounters` or `oncology_timeline_events` if they record regimens. 
   - Gold chunk guidance:
     - Focus on chunks that list active or recent meds (lists, med history sections). 
     - If oncology regimens are recorded in timeline events rather than meds, include those chunks too. 

4. **Oncology timeline**

   - Question:  
     “Summarize the patient’s oncology-related timeline, including major events and treatments.”
   - Primary doc_types:  
     `oncology_timeline_events`, `oncology_timeline`, plus supporting `encounters` or `diagnostic_reports` if needed. 
   - Gold chunk guidance:
     - Choose 3–6 chunks covering:
       - Initial diagnosis/first cancer-related event.
       - Key treatment starts/changes (lines of therapy, major procedures). 
       - Notable response/progression events or critical findings (e.g., scan results summaries). 
     - Try to keep the subset coherent and roughly chronological, so a model could reconstruct a timeline from them.

### Extended 5-question set (adds a synthesis question)

For a richer benchmark, add this fifth question; it forces multi-doc-type reasoning.

5. **Cross-document clinical priorities**

   - Question:  
     “What clinical issues appear to be most important for this patient right now?”
   - Primary doc_types:  
     `conditions`, `medications`, `encounters`, `observations`, `diagnostic_reports`, and `oncology_timeline_events`. 
   - Gold chunk guidance:
     - Pick 4–8 chunks such that:
       - Each chunk contributes a distinct piece of evidence (e.g., a condition, a treatment, a key observation/report). 
       - Together they justify why certain issues are “most important” (e.g., active malignancy on treatment, uncontrolled comorbidity, acute complication). 
     - Explicitly favor chunks that are:
       - Recent in time.
       - Clearly interpretable (e.g., summary sections rather than isolated lab values), unless a lab/report is itself critical.

### How to pick gold_chunk_ids in practice

For each patient and each question:

- Step 1: Filter by doc_type and (if you have it) `is_oncology` flag or recency indicators. 
- Step 2: From the filtered subset, manually inspect and choose only the chunks that:
  - Directly answer the question.
  - Are reasonably self-contained (the model doesn’t need many unrelated chunks to interpret them).
- Step 3: Store:
  - `question_text`
  - `patient_id`
  - `gold_chunk_ids` (list of chunk IDs)
  - Optionally a short `rationale` free-text explaining why these chunks were selected (can be handy for later analysis).

This gives you a consistent scheme across all 9 patients: every patient gets the same 4 core questions, and optionally the 5th synthesis question, with gold_chunk_ids drawn from matching doc_types. 

Do you already have a column that marks “recent” vs “historical” chunks (e.g., encounter date), or should I suggest a simple heuristic for recency using only what’s in your `chunks` table?

In [68]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [70]:
# look at all oncology chunks:

import sqlite3
import pandas as pd
from pathlib import Path

db_path = Path("../data/retrieval/metadata.db")
patient_ids = selected_patient_ids

conn = sqlite3.connect(db_path)

# Multiple patient_ids: build an IN (...) placeholder list.
if not patient_ids:
    chunks_df = pd.DataFrame()  # avoid invalid SQL: IN ()
else:
    placeholders = ",".join(["?"] * len(patient_ids))
    query = f"""
    SELECT
        chunks.patient_id,
        documents.doc_type,
        documents.title,
        chunks.heading,
        chunks.chunk_id,
        chunks.is_oncology,
        chunks.chunk_text
    FROM chunks
    JOIN documents ON chunks.document_id = documents.document_id
    WHERE chunks.patient_id IN ({placeholders})
    """
    chunks_df = pd.read_sql_query(query, conn, params=patient_ids)

conn.close()

display(chunks_df.head())
print("Number of rows:", len(chunks_df))

onc_chunks = chunks_df[chunks_df["is_oncology"] == 1]
display(onc_chunks.head())
print("Number of oncology chunks:", len(onc_chunks))

,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
1,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,cd9bf67f542dee2c5c6eb4e889086745d239ff7b,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
2,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,c6f4af530450ec4d38f7f478693b4d62c2b3467c,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
3,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,9d5bdbca525ce28a75b1ad7deff73853cc1b20eb,0,patient_id: 29f6beee-162f-0113-7884-7224581469...
4,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,250fb82c086a015ec8fe85d5feeb46806e632e86,0,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of rows: 14390


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_text
0,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
21,29f6beee-162f-0113-7884-72245814693f,conditions,conditions.csv,conditions,34e083fbe248b34ab7ff75e989fc91da40a6b511,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
931,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,6c44d78bfe33c70133759592f49f88f9cd26734c,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
932,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,4f181909c2fad14ebbdc20f28f1af14d5dfb89f9,1,patient_id: 29f6beee-162f-0113-7884-7224581469...
933,29f6beee-162f-0113-7884-72245814693f,observations,observations.csv,observations,b02dbcb96bafeb63dc73b3c40a9f184cb0e781bf,1,patient_id: 29f6beee-162f-0113-7884-7224581469...


Number of oncology chunks: 890


In [72]:
onc_chunks["doc_type"].value_counts()

doc_type
oncology_timeline_events    378
procedures                  253
observations                109
oncology_timeline            71
patient_overview             63
conditions                   16
Name: count, dtype: int64

In [76]:
onc_chunks.groupby("patient_id")["doc_type"].count()

patient_id
29f6beee-162f-0113-7884-72245814693f    124
3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678     58
41681ed6-efc5-94c0-1bc0-f60b34dbd31b    116
4736727e-63f4-071a-1516-a49310f5a052    112
aee216e6-cbe8-eaf2-3241-4bd1e8a01494    122
d65197b3-056a-2136-b584-77f43c29da3f     73
ecc4a7d0-8838-36b4-44ba-676d5a1f7927    114
f203e11d-5573-1624-69b8-af8436987b3e    113
f3739580-797d-ae04-eebf-aeddb2fc2f64     58
Name: doc_type, dtype: int64

In [ ]:
onc_chunks.groupby("doc_type")["patient_id"].value_counts()

doc_type                  patient_id                          
conditions                29f6beee-162f-0113-7884-72245814693f     2
                          ecc4a7d0-8838-36b4-44ba-676d5a1f7927     2
                          41681ed6-efc5-94c0-1bc0-f60b34dbd31b     2
                          f203e11d-5573-1624-69b8-af8436987b3e     2
                          aee216e6-cbe8-eaf2-3241-4bd1e8a01494     2
                          f3739580-797d-ae04-eebf-aeddb2fc2f64     2
                          3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678     2
                          d65197b3-056a-2136-b584-77f43c29da3f     1
                          4736727e-63f4-071a-1516-a49310f5a052     1
observations              3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678    14
                          f203e11d-5573-1624-69b8-af8436987b3e    14
                          aee216e6-cbe8-eaf2-3241-4bd1e8a01494    14
                          29f6beee-162f-0113-7884-72245814693f    14
                          41681ed6-efc5-

In [79]:
chunks_df.columns

Index(['patient_id', 'doc_type', 'title', 'heading', 'chunk_id', 'is_oncology',
       'chunk_text'],
      dtype='str')

#### 1. **Patient overview**

This filters candidate chunks for one patient for the Patient Overview question, prioritizing summary-like doc types and creating a readable preview. Filtering with isin(...) and sorting with sort_values(...) are standard Pandas patterns for this kind of review table:

In [80]:
selected_patient_ids

['d65197b3-056a-2136-b584-77f43c29da3f',
 'f3739580-797d-ae04-eebf-aeddb2fc2f64',
 '4736727e-63f4-071a-1516-a49310f5a052',
 '29f6beee-162f-0113-7884-72245814693f',
 '41681ed6-efc5-94c0-1bc0-f60b34dbd31b',
 'aee216e6-cbe8-eaf2-3241-4bd1e8a01494',
 'ecc4a7d0-8838-36b4-44ba-676d5a1f7927',
 '3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678',
 'f203e11d-5573-1624-69b8-af8436987b3e']

In [84]:
import pandas as pd
from IPython.display import display

QUESTION_TYPE = "patient_overview"
QUESTION_TEXT = "Give a concise overview of this patient’s medical background and current care context."
# do this for each patient
#PATIENT_ID = "d65197b3-056a-2136-b584-77f43c29da3f"   # <-- P_1, FIRST OF THE 9 SELECTED PATIENTS
PATIENT_ID = "f3739580-797d-ae04-eebf-aeddb2fc2f64"   # <-- P_2, SECOND OF THE 9 SELECTED PATIENTS


overview_doc_types = [
    "patient_overview",
    "encounters",
    "conditions",
    "oncology_timeline_events",
]

candidate_chunks = chunks_df.loc[
    (chunks_df["patient_id"].astype(str) == str(PATIENT_ID)) &
    (chunks_df["doc_type"].isin(overview_doc_types)),
    [
        "patient_id",
        "doc_type",
        "title",
        "heading",
        "chunk_id",
        "is_oncology",
        "chunk_text",
    ]
].copy()

# Prefer summary-like content first
doc_type_priority = {
    "patient_overview": 0,
    "oncology_timeline_events": 1,
    "conditions": 2,
    "encounters": 3,
}
candidate_chunks["doc_type_priority"] = (
    candidate_chunks["doc_type"].map(doc_type_priority).fillna(99)
)

# Helpful preview for manual review
candidate_chunks["chunk_preview"] = (
    candidate_chunks["chunk_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 400)
)

candidate_chunks = candidate_chunks.sort_values(
    ["doc_type_priority", "doc_type", "title", "heading", "chunk_id"],
    ascending=[True, True, True, True, True]
).reset_index(drop=True)

print("Question:")
print(QUESTION_TEXT)
print("\nCandidate chunks:")
display(
    candidate_chunks[
        [
            "patient_id",
            "doc_type",
            "title",
            "heading",
            "chunk_id",
            "is_oncology",
            "chunk_preview",
        ]
    ]
)

Question:
Give a concise overview of this patient’s medical background and current care context.

Candidate chunks:


,patient_id,doc_type,title,heading,chunk_id,is_oncology,chunk_preview
0,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Patient Overview: Florine959 Stark857,Diagnostic Reports,52403284d08210e63b8f7da16cf841449b40970b,1,- History and physical note; date: 2022-06-21T...
1,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Patient Overview: Florine959 Stark857,Identity,c6527585db111ae98c9a0e1c6f7e2c6310de314f,1,- Patient ID: f3739580-797d-ae04-eebf-aeddb2fc...
2,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Patient Overview: Florine959 Stark857,Medications,be283721d4470a4515d90787d7ce60f5a92db51e,1,- 5 ML fulvestrant 50 MG/ML Prefilled Syringe;...
3,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Patient Overview: Florine959 Stark857,Procedures,c5977a27392b80dbae213d935894d1ca87c37430,1,- Medication Reconciliation (procedure); date:...
4,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Patient Overview: Florine959 Stark857,Provenance,b902b859c7556dc1a4005ee3c9a1e8812c6cdcc1,1,- This document is derived from normalized CSV...
...,...,...,...,...,...,...,...
56,f3739580-797d-ae04-eebf-aeddb2fc2f64,encounters,encounters.csv,encounters,e6601c90b53fbd7d8b4a95ee2c16d1f916782220,0,patient_id: f3739580-797d-ae04-eebf-aeddb2fc2f...
57,f3739580-797d-ae04-eebf-aeddb2fc2f64,encounters,encounters.csv,encounters,ec2fee2226cd6852e65b9504249216c37b9a8cf6,0,patient_id: f3739580-797d-ae04-eebf-aeddb2fc2f...
58,f3739580-797d-ae04-eebf-aeddb2fc2f64,encounters,encounters.csv,encounters,f8c7ba8227e2b3c82f3836200e105809309573e9,0,patient_id: f3739580-797d-ae04-eebf-aeddb2fc2f...
59,f3739580-797d-ae04-eebf-aeddb2fc2f64,encounters,encounters.csv,encounters,fdc99474c72dab7d194529c2e384487b240bde38,0,patient_id: f3739580-797d-ae04-eebf-aeddb2fc2f...


So how do I choose 2-4 chunks for the question Q_1: "Give a concise overview of this patient’s medical background and current care context."
?
See above in [Core 4-question set]
     - Include 2–4 chunks that together cover:
       - Key chronic conditions. 
       - Major past procedures or events if they define the patient’s course. 
       - A high-level oncology context if applicable (e.g., diagnosis, line of therapy). 
     - Prefer chunks that are explicitly summary-like (e.g., overview notes) over raw measurements.

For P_1 we have a summary of recent care in 
`data/derived/sample50/d65197b3-056a-2136-b584-77f43c29da3f/patient_overview.md`
E.g.
```
## Recent Conditions
- Otitis media; date: 2021-06-16T04:22:18-04:00; status: resolved
- Malignant neoplasm of breast (disorder); date: 2021-06-14T04:22:18-04:00; status: active

## Recent Results
- Cancer Disease Progression; value: Patient's condition improved; date: 2022-05-20T18:21:55-04:00
- Treatment status Cancer; value: Treatment changed (situation); date: 2022-05-20T18:21:55-04:00
- Cancer Disease Progression; value: Patient's condition improved; date: 2021-12-06T03:42:59-05:00
- Tobacco smoking status NHIS; value: Never smoker; date: 2021-11-26T03:22:18-05:00
- Respiratory rate; value: 13.0 /min; date: 2021-11-26T03:22:18-05:00
- Heart rate; value: 80.0 /min; date: 2021-11-26T03:22:18-05:00
- Blood Pressure; date: 2021-11-26T03:22:18-05:00
- Head Occipital-frontal circumference; value: 43.41 cm; date: 2021-11-26T03:22:18-05:00
- Weight-for-length Per age and sex; value: 65.134 %; date: 2021-11-26T03:22:18-05:00
- Body Weight; value: 8.7 kg; date: 2021-11-26T03:22:18-05:00
- Pain severity - 0-10 verbal numeric rating [Score] - Reported; value: 2.0 {score}; date: 2021-11-26T03:22:18-05:00
- Body Height; value: 69.8 cm; date: 2021-11-26T03:22:18-05:00

## Recent Encounters
- Encounter for problem; start: 2022-05-20T18:21:55-04:00; end: 2022-05-20T18:36:55-04:00; provider: HALLMARK HEALTH SYSTEM
...
```


So how about these chunks for P_1 Q_1:
in overview:
for recent conditions
45b0f62c2348b66d770620e4e5acfe630cf41f96
for recent results
abc3ffd40cbb13d210ac910a0f5aaa43b998a587
for procedures 
e8429a1cb4b336cbf3be8fa709655ab4d42132e2
for conditions.csv there are two chunks, one has no oncology, one is oncology (I should have chosen both!)
82ad22a90a66f121eaaaa26c95c75d70699b3dbd

So how about these chunks for P_2 Q_1:
in overview:
for recent conditions
4c17e4784d257f85e9fd1ddd74054e5dc3adae9f
for recent results
a9165d0158523ea3b0c882ba42cd1f37f02f067d
for procedures 
c5977a27392b80dbae213d935894d1ca87c37430
for conditions.csv there are 5 chunks and each describes a different condition
...

So we have a pattern:

“All chunks from conditions, plus all chunks whose heading is one of Recent Condition, Recent Results, or Procedures are relevant for ‘Patient Overview’.”


##### If you manually choose the chunks:

1. Save chosen gold chunks

After looking through that table, paste in the 2–4 chunk_id values you want to use as gold chunks.
```python
patient_overview_annotation_for_P1 = pd.DataFrame([{
    "patient_id": PATIENT_ID,
    "question_type": QUESTION_TYPE,
    "question_text": QUESTION_TEXT,
    "gold_chunk_ids": gold_chunk_ids_for_P1_Q1,
}])

display(patient_overview_annotation_for_P1)
```

2. Collect all Patient Overview annotations into a single DataFrame.
Assuming you’ve run the small “annotation” cell multiple times and have a list of per-patient DataFrames (or you re-create them from memory), you can concatenate them:
```python
import pandas as pd
from IPython.display import display

# Suppose for each patient you ran something like:
# patient_overview_annotation_for_X = pd.DataFrame([{
#     "patient_id": PATIENT_ID_X,
#     "question_type": "patient_overview",
#     "question_text": QUESTION_TEXT,
#     "gold_chunk_ids": gold_chunk_ids_for_X,
# }])

patient_overview_annotations = pd.concat(
    [
        patient_overview_annotation_for_P1,
        patient_overview_annotation_for_P2,
        # ...
        patient_overview_annotation_for_P9,
    ],
    ignore_index=True,
)

display(patient_overview_annotations)
```

3. Persist this as part of your evaluation set (e.g., CSV).
Store the annotations so you can reuse them when you actually run retrieval experiments.
```python
patient_overview_annotations.to_csv(
    "data/interim/eval_patient_overview_annotations.csv",
    index=False,
)
```
Later, when you run your retrieval pipeline, the workflow is:
For each row:
- Use patient_id and question_text to form the query.
- Retrieve top N chunks.
- Compare retrieved chunk_ids against gold_chunk_ids (hit rate, recall@k, etc.).

In [ ]:
# gold_chunk_ids_for_P1_Q1 = [
#     # paste chosen chunk_ids here
#     "45b0f62c2348b66d770620e4e5acfe630cf41f96",
#     "abc3ffd40cbb13d210ac910a0f5aaa43b998a587",
#     "e8429a1cb4b336cbf3be8fa709655ab4d42132e2",
#     "82ad22a90a66f121eaaaa26c95c75d70699b3dbd"
# ]

# patient_overview_annotation_for_P1 = pd.DataFrame([{
#     "patient_id": PATIENT_ID,
#     "question_type": QUESTION_TYPE,
#     "question_text": QUESTION_TEXT,
#     "gold_chunk_ids": gold_chunk_ids_for_P1_Q1,
# }])

# display(patient_overview_annotation_for_P1)

,patient_id,question_type,question_text,gold_chunk_ids
0,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,Give a concise overview of this patient’s medi...,"[45b0f62c2348b66d770620e4e5acfe630cf41f96, abc..."


##### If selecting a pattern:


In [85]:
import pandas as pd
from IPython.display import display

QUESTION_TYPE = "patient_overview"
QUESTION_TEXT = "Give a concise overview of this patient’s medical background and current care context."

overview_doc_types = ["conditions"]
overview_headings = [
    "Recent Condition",
    "Recent Results",
    "Procedures",
]

# Boolean mask for relevant chunks according to your rule
mask_conditions = chunks_df["doc_type"] == "conditions"
mask_headings = chunks_df["heading"].isin(overview_headings)

relevant_mask = mask_conditions | mask_headings

patient_overview_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

patient_overview_gold["question_type"] = QUESTION_TYPE
patient_overview_gold["question_text"] = QUESTION_TEXT

# Optional: reorder columns for clarity
patient_overview_gold = patient_overview_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(patient_overview_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,patient_overview,Give a concise overview of this patient’s medi...,"[82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8, cd9..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,patient_overview,Give a concise overview of this patient’s medi...,"[3a5cb52256a2e8b429454dc59790f4566d3be9e6, b34..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,patient_overview,Give a concise overview of this patient’s medi...,"[973694e5061b9103d1b1da41a6fb44459e220b91, 55b..."
3,4736727e-63f4-071a-1516-a49310f5a052,patient_overview,Give a concise overview of this patient’s medi...,"[66d329c2430b7e3328c7e0841eee78ffe79e718a, a76..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,patient_overview,Give a concise overview of this patient’s medi...,"[8e2f0f0d9e606624f71d9436e41488245e902e82, c34..."
5,d65197b3-056a-2136-b584-77f43c29da3f,patient_overview,Give a concise overview of this patient’s medi...,"[5c819ccda0a7fa90e3bd68908860eeccc7df7be4, 82a..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,patient_overview,Give a concise overview of this patient’s medi...,"[aa4ee09b468f01436be3e9173c0a260db67fc125, 639..."
7,f203e11d-5573-1624-69b8-af8436987b3e,patient_overview,Give a concise overview of this patient’s medi...,"[e185780dd0261fb16e25761be2679bf4ff238343, 8d2..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,patient_overview,Give a concise overview of this patient’s medi...,"[ebc2298acced3daa29baa223801bbcf435a73627, a2c..."


#### 2. **Conditions**

   - Question:  
     “What are the patient’s main diagnosed conditions?”
   - Primary doc_types:  
     `conditions`, possibly `encounters` or `patient_overview` if they contain canonical lists. 
   - Gold chunk guidance:
     - Select chunks that list named diagnoses (problem lists, diagnosis sections, structured condition records). 
     - If conditions evolve (e.g., disease progression or resolved conditions), include chunks that clearly mark the current status.

==> Looks like we need all chunks from conditions.csv, and from patient_overview.md Recent Conditions, Recent Encounters, Recent Results.

You’re right: those Recent Results examples are clearly condition-related, and for your “main diagnosed conditions” question they’re complementary, not noise.

Conceptually:

Recent Condition chunks list what the conditions are and their status.

Recent Results chunks describe status, staging, and treatment changes, which are part of “main diagnosed conditions” in an oncology context.

In [88]:
QUESTION_TYPE = "conditions"
QUESTION_TEXT = "What are the patient’s main diagnosed conditions?"

condition_doc_types = ["conditions"]
condition_headings = [
    "Recent Condition",
    "Recent Results",
]

mask_doc_type = chunks_df["doc_type"].isin(condition_doc_types)
mask_heading = chunks_df["heading"].isin(condition_headings)

relevant_mask = mask_doc_type | mask_heading

conditions_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

conditions_gold["question_type"] = QUESTION_TYPE
conditions_gold["question_text"] = QUESTION_TEXT

conditions_gold = conditions_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(conditions_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,conditions,What are the patient’s main diagnosed conditions?,"[82fbc5fb0df955ed9b5861c956dd9d5cdd0a1aa8, cd9..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,conditions,What are the patient’s main diagnosed conditions?,"[3a5cb52256a2e8b429454dc59790f4566d3be9e6, b34..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,conditions,What are the patient’s main diagnosed conditions?,"[973694e5061b9103d1b1da41a6fb44459e220b91, 55b..."
3,4736727e-63f4-071a-1516-a49310f5a052,conditions,What are the patient’s main diagnosed conditions?,"[66d329c2430b7e3328c7e0841eee78ffe79e718a, a76..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,conditions,What are the patient’s main diagnosed conditions?,"[8e2f0f0d9e606624f71d9436e41488245e902e82, c34..."
5,d65197b3-056a-2136-b584-77f43c29da3f,conditions,What are the patient’s main diagnosed conditions?,"[5c819ccda0a7fa90e3bd68908860eeccc7df7be4, 82a..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,conditions,What are the patient’s main diagnosed conditions?,"[aa4ee09b468f01436be3e9173c0a260db67fc125, 639..."
7,f203e11d-5573-1624-69b8-af8436987b3e,conditions,What are the patient’s main diagnosed conditions?,"[e185780dd0261fb16e25761be2679bf4ff238343, 8d2..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,conditions,What are the patient’s main diagnosed conditions?,"[ebc2298acced3daa29baa223801bbcf435a73627, a2c..."


#### 3. **Medications**

   - Question:  
     “What medications is the patient taking or has recently taken?”
   - Primary doc_types:  
     `medications`, optionally `encounters` or `oncology_timeline_events` if they record regimens. 
   - Gold chunk guidance:
     - Focus on chunks that list active or recent meds (lists, med history sections). 
     - If oncology regimens are recorded in timeline events rather than meds, include those chunks too. 


===> There's Medication heading in the overview. 

you want to focus on chunks that actually name medications or regimens, or at least clearly describe treatment as “medication use,” not generic procedures.

Recommended rule for this question
Based on your concrete data:

Always include:

doc_type == "medications" chunks.
These are your structured med list.

patient_overview chunks with a heading that clearly corresponds to medications (e.g., "Medications").
These give a concise summary of the meds.

In [ ]:
chunks_df

Index(['patient_id', 'doc_type', 'title', 'heading', 'chunk_id', 'is_oncology',
       'chunk_text'],
      dtype='str')

In [90]:
chunks_df["doc_type"].value_counts()

doc_type
observations                5508
procedures                  3679
diagnostic_reports          2201
encounters                  1633
conditions                   466
oncology_timeline_events     378
medications                  373
patient_overview              81
oncology_timeline             71
Name: count, dtype: int64

In [91]:
QUESTION_TYPE = "medications"
QUESTION_TEXT = "What medications is the patient taking or has recently taken?"

med_doc_types = ["medications"]
med_headings = ["Medications"]   # adjust to exact heading text in patient_overview.md

mask_doc_type = chunks_df["doc_type"].isin(med_doc_types)
mask_heading = (
    (chunks_df["doc_type"] == "patient_overview") &
    (chunks_df["heading"].isin(med_headings))
)

relevant_mask = mask_doc_type | mask_heading

medications_gold = (
    chunks_df.loc[relevant_mask, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

medications_gold["question_type"] = QUESTION_TYPE
medications_gold["question_text"] = QUESTION_TEXT

medications_gold = medications_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(medications_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,medications,What medications is the patient taking or has ...,"[5304dd221e415c2eb3ca37fbf297c4c9df22b782, 621..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,medications,What medications is the patient taking or has ...,"[d758fffebfaf17a398ca6c13bfe747a4b09596f4, 6e8..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,medications,What medications is the patient taking or has ...,"[ec58e0dc34d7697a640afa5b7f81bd159bd5c743, d04..."
3,4736727e-63f4-071a-1516-a49310f5a052,medications,What medications is the patient taking or has ...,"[44c1260f6598a7aa5cb578f22d739d94c396b3cc, dba..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,medications,What medications is the patient taking or has ...,"[37a97c87dc3070e5388c09b44651676450ed54b0, 3d6..."
5,d65197b3-056a-2136-b584-77f43c29da3f,medications,What medications is the patient taking or has ...,"[3dd64d3fe22d27dd201b300103caff9732a50956, 506..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,medications,What medications is the patient taking or has ...,"[def6d970e277eb18ffd3c881f3d437e9d124ef45, 104..."
7,f203e11d-5573-1624-69b8-af8436987b3e,medications,What medications is the patient taking or has ...,"[4652c18e53dba09dfa80671419f043105165869a, 265..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,medications,What medications is the patient taking or has ...,"[b442d7ffecc3eb3195ee98d7f3ba04f1aaee4e7b, 824..."


#### 4. **Oncology timeline**

   - Question:  
     “Summarize the patient’s oncology-related timeline, including major events and treatments.”
   - Primary doc_types:  
     `oncology_timeline_events`, `oncology_timeline`, plus supporting `encounters` or `diagnostic_reports` if needed. 
   - Gold chunk guidance:
     - Choose 3–6 chunks covering:
       - Initial diagnosis/first cancer-related event.
       - Key treatment starts/changes (lines of therapy, major procedures). 
       - Notable response/progression events or critical findings (e.g., scan results summaries). 
     - Try to keep the subset coherent and roughly chronological, so a model could reconstruct a timeline from them.


Given your data:

oncology_timeline_events has many granular entries (30+ per patient).

oncology_timeline has far fewer (e.g., 9 chunks) and is explicitly designed as a patient-level timeline summary.

patient_overview.md has Recent Conditions, which give high-level disease context but not a full chronological narrative.

I would:

Include all chunks with doc_type == "oncology_timeline".
These are your primary gold chunks: they encode the backbone of the cancer journey.

Optionally include Recent Condition chunks from patient_overview only if they add important context that’s missing from oncology_timeline, e.g.:

First mention of a malignancy that doesn’t clearly appear in the timeline summary.

Major comorbid oncologic conditions that help interpret the timeline (e.g., multiple primaries).

In [92]:
QUESTION_TYPE = "oncology_timeline"
QUESTION_TEXT = (
    "Summarize the patient’s oncology-related timeline, including major events and treatments."
)

timeline_doc_types = ["oncology_timeline"]

mask_timeline = chunks_df["doc_type"].isin(timeline_doc_types)

oncology_timeline_gold = (
    chunks_df.loc[mask_timeline, ["patient_id", "chunk_id"]]
    .groupby("patient_id")["chunk_id"]
    .apply(list)
    .reset_index()
    .rename(columns={"chunk_id": "gold_chunk_ids"})
)

oncology_timeline_gold["question_type"] = QUESTION_TYPE
oncology_timeline_gold["question_text"] = QUESTION_TEXT

oncology_timeline_gold = oncology_timeline_gold[
    ["patient_id", "question_type", "question_text", "gold_chunk_ids"]
]

display(oncology_timeline_gold)

,patient_id,question_type,question_text,gold_chunk_ids
0,29f6beee-162f-0113-7884-72245814693f,oncology_timeline,Summarize the patient’s oncology-related timel...,"[c21e06064148b6da08db071c8263d1da295ed93c, 17f..."
1,3a1c7c7b-0f87-e7ba-f2ed-1b0882fe3678,oncology_timeline,Summarize the patient’s oncology-related timel...,"[22d46b9aab128e2a1acdc0d26d872f8bd10f7f79, 674..."
2,41681ed6-efc5-94c0-1bc0-f60b34dbd31b,oncology_timeline,Summarize the patient’s oncology-related timel...,"[e5ca0cdb0930f81b8adfb7fa94b2b6cb38539862, 441..."
3,4736727e-63f4-071a-1516-a49310f5a052,oncology_timeline,Summarize the patient’s oncology-related timel...,"[5548c8e86138b4a33e1d8a12c518e43dcd667f50, 441..."
4,aee216e6-cbe8-eaf2-3241-4bd1e8a01494,oncology_timeline,Summarize the patient’s oncology-related timel...,"[30263ea4c834c2be4745a107b2f8386cbd98a1e4, 2fe..."
5,d65197b3-056a-2136-b584-77f43c29da3f,oncology_timeline,Summarize the patient’s oncology-related timel...,"[0ab557068bb85b16a4aad4d40f2a6c83cd45d5b3, 1fb..."
6,ecc4a7d0-8838-36b4-44ba-676d5a1f7927,oncology_timeline,Summarize the patient’s oncology-related timel...,"[4105225d21ffe94cb84279be1b62e59149f4ed35, fde..."
7,f203e11d-5573-1624-69b8-af8436987b3e,oncology_timeline,Summarize the patient’s oncology-related timel...,"[c9c3776fbdbf55e34536d8b71d32ba80b80f0ad2, 98d..."
8,f3739580-797d-ae04-eebf-aeddb2fc2f64,oncology_timeline,Summarize the patient’s oncology-related timel...,"[d963a6518333e3e87c905fcecd599bf3f90c3f21, 330..."


In [ ]:
# # to include Recent Condition

# timeline_doc_types = ["oncology_timeline"]
# condition_headings = ["Recent Condition"]

# mask_timeline = chunks_df["doc_type"].isin(timeline_doc_types)
# mask_recent_condition = (
#     (chunks_df["doc_type"] == "patient_overview") &
#     (chunks_df["heading"].isin(condition_headings))
# )

# relevant_mask = mask_timeline | mask_recent_condition

## Retrieval evaluation for 4 questions

Yes, you already have enough structure to start running a **meaningful retrieval evaluation** with your first 4 questions, even without question 5.

You now have, per patient:

1. **Patient Overview**  
   - Question: fixed text.  
   - Gold chunks: conditions + recent results/conditions/procedures, per your rule.  
2. **Conditions**  
   - Question: fixed text.  
   - Gold chunks: `conditions` + `Recent Condition` + `Recent Results`.  
3. **Medications**  
   - Question: fixed text.  
   - Gold chunks: `medications` + overview Medications heading.  
4. **Oncology timeline**  
   - Question: fixed text.  
   - Gold chunks: all `oncology_timeline` chunks.  

That’s a set of **4 question types × 9 patients = 36 query–gold pairs**, which is plenty to start seeing whether your retrieval stack is behaving sensibly. In RAG and IR practice, even a few dozen carefully labeled queries can be used to compute recall@k, hit rate, and MRR as an initial “sanity check” evaluation. [vivedhaelango.substack](https://vivedhaelango.substack.com/p/lesson-112-how-do-you-build-a-golden)

What you don’t have yet:

- A cross-document synthesis question (your planned question 5), which is useful but not strictly necessary for a first round.
- A large golden dataset; but early RAG guides explicitly recommend starting with tens of high-quality examples. [menuagentic](https://menuagentic.com/deep-dives/retrieval-and-rag/evaluating-rag)

Suggested next step

You can now:

1. Save each question-type’s gold set as a CSV (like you did for overview) so you have a consolidated evaluation dataset.
2. Implement a small evaluation loop that:
   - For each `(patient_id, question_text)` pair:
     - Runs your retrieval.
     - Compares retrieved `chunk_id`s to `gold_chunk_ids`.  
   - Computes recall@k or hit rate@k for each question type. [asoasis](https://asoasis.tech/articles/2026-05-04-0853-retrieval-augmented-generation-evaluation-metrics/)
3. Inspect per-question-type performance to see where retrieval is strong or weak (e.g., maybe meds retrieval is weaker than timeline).

You can always add question 5 later, but you definitely have enough labeled structure to start running and learning from an evaluation now.

Do you want a compact example of a retrieval-eval loop in pseudocode/Python that assumes a function like `retrieve_chunks(patient_id, question_text, k)` and uses your gold CSVs?



In [93]:
import math

def eval_search_type_df(gold_df, search_type, k=5):
    """
    Evaluate a given search_type over all rows in a gold DataFrame.

    gold_df columns:
      - patient_id
      - question_text  (or 'question' if you prefer)
      - gold_chunk_ids (list of chunk_ids)
    """
    hits = []
    mrrs = []

    for _, row in gold_df.iterrows():
        patient_id = row["patient_id"]
        question = row["question_text"]   # matches your patient_overview_gold schema
        gold_ids = row["gold_chunk_ids"]

        retrieved_ids = get_ranked_chunk_ids(
            query=question,
            patient_id=patient_id,
            search_type=search_type,
            k=k,
        )

        hits.append(hit_rate_at_k(retrieved_ids, gold_ids, k))
        mrrs.append(mrr_at_k(retrieved_ids, gold_ids, k))

    hit_rate = sum(hits) / len(hits) if hits else math.nan
    mrr = sum(mrrs) / len(mrrs) if mrrs else math.nan

    return hit_rate, mrr

In [94]:
# Filter just the patient_overview rows if you have multiple question_types
po_gold = patient_overview_gold[
    patient_overview_gold["question_type"] == "patient_overview"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[patient_overview] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[patient_overview] lexical: Hit@5 = 0.778, MRR@5 = 0.189
[patient_overview] semantic: Hit@5 = 1.000, MRR@5 = 0.289
[patient_overview] hybrid: Hit@5 = 1.000, MRR@5 = 0.313


These numbers say your **semantic and hybrid retrieval are clearly better than lexical** for the Patient Overview question type, and hybrid is doing the best job at ranking the relevant chunks near the top. [medium](https://medium.com/@rajnish_khatri/retrieval-metrics-tutorial-recall-k-and-mrr-explained-d2f12afb9c89)

Here’s the breakdown:

Hit@5

- **Lexical: 0.778**  
  For about 7 out of 9 patients, at least one gold chunk appears in the top 5; for 2 patients, *no* relevant chunk is in the top 5. [apxml](https://apxml.com/courses/getting-started-rag/chapter-6-evaluating-improving-rag-systems/evaluating-retrieval)
- **Semantic: 1.000**  
  For all 9 patients, at least one gold chunk is in the top 5 — semantic search never completely misses on this question type. [youtube](https://www.youtube.com/watch?v=-dIm4EyUrrM)
- **Hybrid: 1.000**  
  Same as semantic: every query has a hit within the top 5.

Interpretation:

- Lexical retrieval sometimes fails to surface any overview/conditions/results/procedures chunk in the top-5, probably because the wording of the Patient Overview question doesn’t match the chunk text strongly enough for those patients. [ludekkvapil](https://ludekkvapil.cz/skills/rag/)
- Semantic and hybrid retrieval are **reliably finding at least one relevant chunk** in the top-5 for all patients, which is exactly what you want from a baseline retriever for these overview questions. [medium](https://medium.com/@rajnish_khatri/retrieval-metrics-tutorial-recall-k-and-mrr-explained-d2f12afb9c89)

MRR@5

Remember: MRR is roughly \(1 / \text{average rank of the first hit in the top-5}\). [en.wikipedia](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)

- **Lexical: 0.189**  
  Average first-hit rank is around \(1 / 0.189 \approx 5.3\).  
  So even when lexical finds a relevant chunk, it’s often **near the bottom of the top-5** (or missing in some queries, which contributes 0). [metricgate](https://metricgate.com/blogs/mean-reciprocal-rank-search-evaluation/)
- **Semantic: 0.289**  
  Average first-hit rank is about \(1 / 0.289 \approx 3.5\).  
  Typically the first relevant chunk appears around rank 3–4. [en.wikipedia](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)
- **Hybrid: 0.313**  
  Average first-hit rank is \(1 / 0.313 \approx 3.2\).  
  On average, hybrid pushes the **first relevant chunk slightly closer to the top** than semantic, so the model will see good context earlier. [metricgate](https://metricgate.com/docs/mean-reciprocal-rank-mrr/)

Interpretation:

- Lexical is not only less reliable, it also tends to bury the relevant chunk lower in the ranking. [fabianhertwig](https://fabianhertwig.com/blog/information-retrieval-metrics/)
- Semantic is better for both **coverage** (Hit@5) and **ranking quality** (MRR): relevant chunks show up earlier in the list. [ludekkvapil](https://ludekkvapil.cz/skills/rag/)
- Hybrid improves ranking further over semantic, which is exactly what you’d hope for when combining lexical and semantic signals. [medium](https://medium.com/@rajnish_khatri/retrieval-metrics-tutorial-recall-k-and-mrr-explained-d2f12afb9c89)

What this suggests for your pipeline

For **Patient Overview** questions:

- You should probably **prefer hybrid retrieval** as your default — it:
  - Never misses within top-5. [apxml](https://apxml.com/courses/getting-started-rag/chapter-6-evaluating-improving-rag-systems/evaluating-retrieval)
  - Delivers relevant chunks relatively high in the ranking.
- Semantic alone is still a strong option; lexical alone is clearly weaker and may be best kept as a baseline or fallback. [ludekkvapil](https://ludekkvapil.cz/skills/rag/)

A good next step:

- Run the same evaluation for your other three question types (Conditions, Medications, Oncology Timeline) and see if the pattern holds:
  - If hybrid consistently wins on Hit@5 and MRR@5, that’s a strong argument to standardize on hybrid for your first production-style experiments. [ragflow](https://ragflow.io/docs/run_retrieval_test)
  - If you see differences (e.g., lexical doing better on very structured questions), you could consider per-question-type retrieval strategies.

Would you like guidance on choosing K (e.g., comparing Hit@3 and Hit@10) or adding a simple Precision@k to check how “clean” the top results are?

In [95]:
# Filter just the conditions rows if you have multiple question_types
po_gold = conditions_gold[
    conditions_gold["question_type"] == "conditions"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[conditions_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[conditions_gold] lexical: Hit@5 = 1.000, MRR@5 = 1.000
[conditions_gold] semantic: Hit@5 = 0.778, MRR@5 = 0.289
[conditions_gold] hybrid: Hit@5 = 1.000, MRR@5 = 0.748


In [ ]:
# Filter just the medications rows if you have multiple question_types
po_gold = medications_gold[
    medications_gold["question_type"] == "medications"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[medications_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[medications_gold] lexical: Hit@5 = 1.000, MRR@5 = 1.000
[medications_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.741
[medications_gold] hybrid: Hit@5 = 1.000, MRR@5 = 1.000


In [97]:
# Filter just the oncology_timeline rows if you have multiple question_types
po_gold = oncology_timeline_gold[
    oncology_timeline_gold["question_type"] == "oncology_timeline"
]

for st in ["lexical", "semantic", "hybrid"]:
    hr, mrr = eval_search_type_df(po_gold, st, k=5)
    print(f"[oncology_timeline_gold] {st}: Hit@5 = {hr:.3f}, MRR@5 = {mrr:.3f}")

[oncology_timeline_gold] lexical: Hit@5 = 1.000, MRR@5 = 1.000
[oncology_timeline_gold] semantic: Hit@5 = 1.000, MRR@5 = 0.944
[oncology_timeline_gold] hybrid: Hit@5 = 1.000, MRR@5 = 1.000


Yes, these results are consistent with the way you defined gold chunks for these three question types, and the 1.000 values are largely expected. They also tell you something useful about **which search mode is best for which kind of question**. [medium](https://medium.com/@rajnish_khatri/retrieval-metrics-tutorial-recall-k-and-mrr-explained-d2f12afb9c89)

## Why so many 1.000s?

Recall what Hit@5 and MRR@5 mean: [en.wikipedia](https://en.wikipedia.org/wiki/Mean_reciprocal_rank)

- **Hit@5 = 1.000**  
  For all 9 patients, at least one gold chunk appears in the top 5.
- **MRR@5 = 1.000**  
  For all 9 patients, the **first relevant chunk is at rank 1** (top result) — every query’s top-1 is a gold chunk.

Given your labeling rules:

- **Conditions question**  
  Gold = all `conditions` chunks + `Recent Condition` + `Recent Results`.  
  These are highly structured, diagnosis-centric chunks with text matching the question (“conditions”, disease names, statuses), so lexical search is very likely to put them right at the top. [cl.cam.ac](https://www.cl.cam.ac.uk/teaching/1718/InfoRtrv/slides/lecture6-evaluation.pdf)
- **Medications question**  
  Gold = `medications` doc_type + Medications heading in `patient_overview`.  
  Again, highly structured, name-heavy content where lexical signals (drug names) dominate, making it easy for lexical (and hybrid) search to put them at rank 1. [build.fhir](https://build.fhir.org/ig/HL7/US-Core/medication-list.html)
- **Oncology timeline question**  
  Gold = all `oncology_timeline` chunks.  
  Because you have a dedicated timeline doc_type, those chunks likely contain characteristic oncology events and headings that match the question, and you probably have fewer competing chunks in that doc_type per patient, so lexical and hybrid retrieval have a straightforward target. [aclanthology](https://aclanthology.org/2024.clinicalnlp-1.53/)

So yes: for these more **structured, narrow questions** with tight, schema-driven labels, it’s not surprising that:

- Lexical and hybrid retrieval get **Hit@5 = 1.000** and often **MRR@5 = 1.000**, because the “obvious” chunks are easy to match and rank first. [apxml](https://apxml.com/courses/getting-started-rag/chapter-6-evaluating-improving-rag-systems/evaluating-retrieval)

## Interpreting each block

### Conditions

- Lexical: Hit@5 = 1.000, MRR@5 = 1.000  
  Always finds a relevant conditions chunk at rank 1.
- Semantic: Hit@5 = 0.778, MRR@5 = 0.289  
  Sometimes misses in top-5, and when it hits, the first hit is around rank 3–4 on average. [scribd](https://www.scribd.com/presentation/982892265/Semantic-Search-Metrics)
- Hybrid: Hit@5 = 1.000, MRR@5 = 0.748  
  Always hits, usually near the top, but still not as consistently rank-1 as pure lexical.

Interpretation:

- For **conditions**, lexical retrieval is clearly best; semantic alone struggles more with the structure and vocabulary of condition lists. [ludekkvapil](https://ludekkvapil.cz/skills/rag/)
- Hybrid improves over semantic but doesn’t beat pure lexical; likely the lexical signal dominates for this structured question.

### Medications

- Lexical: Hit@5 = 1.000, MRR@5 = 1.000  
- Semantic: Hit@5 = 1.000, MRR@5 = 0.741  
- Hybrid: Hit@5 = 1.000, MRR@5 = 1.000  

Interpretation:

- All modes always retrieve at least one relevant chunk in top-5.
- Lexical and hybrid both put a relevant med chunk at rank 1 for every query; semantic is good but slightly weaker in ranking. [medium](https://medium.com/@rajnish_khatri/retrieval-metrics-tutorial-recall-k-and-mrr-explained-d2f12afb9c89)
- For **medications**, any of the three works, but lexical/hybrid are ideal.

### Oncology timeline

- Lexical: Hit@5 = 1.000, MRR@5 = 1.000  
- Semantic: Hit@5 = 1.000, MRR@5 = 0.944  
- Hybrid: Hit@5 = 1.000, MRR@5 = 1.000  

Interpretation:

- All modes always hit, and even semantic typically puts a relevant timeline chunk very high (MRR ~0.94 means first hit around rank 1–2). [metricgate](https://metricgate.com/docs/mean-reciprocal-rank-mrr/)
- For **oncology timeline**, lexical and hybrid are again extremely strong; semantic is nearly as good.

## Putting this together with Patient Overview

Recall:

- Patient Overview:  
  - Lexical: Hit@5 = 0.778, MRR@5 = 0.189  
  - Semantic: Hit@5 = 1.000, MRR@5 = 0.289  
  - Hybrid: Hit@5 = 1.000, MRR@5 = 0.313

So across all four questions:

- **Patient Overview (broad, narrative)**  
  → hybrid (or semantic) best; lexical sometimes misses and ranks low. [apxml](https://apxml.com/courses/getting-started-rag/chapter-6-evaluating-improving-rag-systems/evaluating-retrieval)
- **Conditions (structured)**  
  → lexical clearly best; hybrid close; semantic weaker. [cl.cam.ac](https://www.cl.cam.ac.uk/teaching/1718/InfoRtrv/slides/lecture6-evaluation.pdf)
- **Medications (structured)**  
  → lexical/hybrid best; semantic good. [build.fhir](https://build.fhir.org/medications-module.html)
- **Oncology timeline (structured + curated summary)**  
  → lexical/hybrid best; semantic almost as good. [arxiv](https://arxiv.org/html/2605.15168v1)

## Practical takeaway for your pipeline

You now have evidence-based guidance:

- Use **hybrid or semantic** for **broad summary questions** like Patient Overview, where semantics matter more than exact token matching. [ludekkvapil](https://ludekkvapil.cz/skills/rag/)
- Use **lexical (or hybrid with strong lexical component)** for **structured schema questions** (conditions, medications, timeline), where names and codes drive relevance. [fabianhertwig](https://fabianhertwig.com/blog/information-retrieval-metrics/)

A simple production rule could be:

- If `question_type` in `{conditions, medications, oncology_timeline}` → default to **lexical** or **hybrid**.
- If `question_type == "patient_overview"` → default to **hybrid** (or semantic if you prefer).  

You can refine this further later, but this is already a solid, data-backed split. [ragflow](https://ragflow.io/docs/run_retrieval_test)

If you want, we can now look at **Hit@k and MRR@k at different k (e.g., 3, 10)** to see how sensitive your retrieval is to the context window size you plan to use.

# NEGATIVE QUESTIONS

# END

After the pipeline works, run one final pass on the full 258-patient corpus to report scalability or at least show that the same code works end-to-end.[confluence.hl7]

A useful pattern is to keep:
patients_50/ for development,
patients_258/ for final scalability testing,
and a manifest file documenting the chosen IDs and selection criteria.
